# NucleiSky3D Benchmarking

## 🎯 Objective
This notebook benchmarks **NucleiSky3D** by repeatedly:
1. Taking **random 3D subvolumes** (patches) from a large reference volume.
2. Running NucleiSky3D to re-localize each patch inside the full volume.
3. Recording accuracy, robustness, and runtime across **patch sizes** and **matchers**.

It mirrors the logic of the 2D benchmarking notebook, but for the 3D pipeline.

---

## ✅ What you get
- **Comparable benchmark table** (CSV) across matchers and patch shapes
- Translation error (µm + voxel units), inlier fraction, mean registration error
- Optional intensity similarity check (SSIM on XY MIPs)
- Runtime profiling
- “Minimum nuclei required” analysis (success vs number of nuclei)

> **Tip:** For fair comparison, this notebook ensures *each matcher sees the same set of patch origins* per patch shape.


# 0. Install + import functions

In [ ]:
# @title  Initialize Environment & Install Dependencies

import os
import sys
import torch
from pathlib import Path

# Metadata
current_version = "0.0.6"
notebook_name = "NucleiSky3DBenchmarking"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    !pip install -q "cellpose[all]" tifffile
    !pip install -q instanseg-torch

    from google.colab import userdata

    # 1. Retrieve the token securely from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        print("Error: Secret 'GITHUB_TOKEN' not found. Please add it in the sidebar (🔑).")
        token = None

    # 2. Install from your private repo
    if token:
        user = 'CellMigrationLab'  # Replace with your GitHub username
        repo = 'NucleiSky'

        # The --upgrade flag ensures you always get the latest version if you re-run
        !pip install --upgrade git+https://{token}@github.com/{user}/{repo}.git

    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")

else:
    # Fallback for local environments
    print("done")


# ---------------------------------------------------------
# 2. GPU Check
# ---------------------------------------------------------
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: No GPU detected. Segmentation with Cellpose or InstanSeg will be VERY SLOW on CPU.")
    print("    -> In Colab, go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")



In [ ]:
#@title Cell 1: 3D benchmark functions

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------

from IPython.display import display, clear_output
from pathlib import Path
import os
import time
import zlib
import json
import copy
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as ssim

# NucleiSky3D imports
from nucleisky.nucleisky3d.pipeline import NucleiSky3D, run_adaptive_nucleisky_3d
from nucleisky.nucleisky3d.config import DEFAULT_MATCHER_CONFIG
from nucleisky.nucleisky3d.preprocess import ij_percentile_normalize


# ---------------------------------------------------------
# Benchmark table columns
# ---------------------------------------------------------

BENCH_COLS_3D = [
    "matcher",

    # Patch geometry
    "patch_z_px",
    "patch_y_px",
    "patch_x_px",
    "patch_shape_label",
    "patch_volume_px",
    "patch_xy_area_px",
    "patch_z0",
    "patch_y0",
    "patch_x0",

    # Query content
    "n_nuclei",

    # Matcher outputs
    "success",
    "frac_inliers",
    "mean_error_um",
    "best_scale",
    "estimated_rotation_deg",

    # Translation error
    "z_error_um",
    "y_error_um",
    "x_error_um",
    "trans_error_um",
    "z_error_px",
    "y_error_px",
    "x_error_px",
    "trans_error_px",

    # Image-level diagnostic
    "ssim_xy",

    # Runtime
    "time_matcher_s",
    "time_total_s",

    # Diagnostics
    "failure_reason",
]


# ---------------------------------------------------------
# General helpers
# ---------------------------------------------------------

def _as_tuple3_3d(v, name="value"):
    """
    Validate and return a length-3 tuple of positive floats.
    Used for voxel size in Z,Y,X order.
    """
    if v is None:
        raise ValueError(f"{name} is missing.")

    vals = tuple(float(x) for x in v)

    if len(vals) != 3:
        raise ValueError(f"{name} must contain exactly 3 values in Z,Y,X order.")

    if any(x <= 0 for x in vals):
        raise ValueError(f"{name} values must be positive.")

    return vals


def _normalise_matcher_name_3d(matcher):
    """
    Normalise 3D matcher names.
    """
    matcher = str(matcher).strip().lower()

    aliases = {
        "hash": "hashing",
        "hashing3d": "hashing",
        "geometric_hashing": "hashing",
        "geometric hashing": "hashing",
        "pyramid3d": "pyramid",
        "tetrahedron": "pyramid",
        "tetrahedra": "pyramid",
        "adaptive3d": "adaptive",
    }

    return aliases.get(matcher, matcher)


def _stable_u32_from_str_3d(s):
    """
    Stable integer hash across Python sessions.
    """
    return zlib.adler32(str(s).encode("utf-8")) & 0xFFFFFFFF


def _deep_merge_dict_3d(base, override):
    """
    Recursively merge override into base without mutating either.
    """
    if override is None:
        return copy.deepcopy(base)

    out = copy.deepcopy(base)

    for k, v in override.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge_dict_3d(out[k], v)
        else:
            out[k] = copy.deepcopy(v)

    return out


def _clip_origin_zyx(z0, y0, x0, Z, Y, X, pz, py, px):
    """
    Clip a 3D patch origin so that the patch remains inside the volume.
    """
    z0 = int(max(0, min(int(z0), int(Z - pz))))
    y0 = int(max(0, min(int(y0), int(Y - py))))
    x0 = int(max(0, min(int(x0), int(X - px))))
    return z0, y0, x0


def _count_nuclei_in_patch_3d(zs_px, ys_px, xs_px, z0, y0, x0, pz, py, px):
    """
    Count centroids inside a ZYX patch.
    """
    z1 = z0 + pz
    y1 = y0 + py
    x1 = x0 + px

    return int(np.sum(
        (zs_px >= z0) & (zs_px < z1) &
        (ys_px >= y0) & (ys_px < y1) &
        (xs_px >= x0) & (xs_px < x1)
    ))


def _rotation_angle_from_identity_deg(R):
    """
    Return the rotation angle, in degrees, between a 3D rotation matrix and identity.

    In this benchmark, the ground-truth crop is not rotated, so this value is an
    estimated rotation magnitude rather than a general rotation-error metric.
    """
    if R is None:
        return np.nan

    try:
        R = np.asarray(R, dtype=float).reshape(3, 3)
        c = (np.trace(R) - 1.0) / 2.0
        c = float(np.clip(c, -1.0, 1.0))
        return float(np.degrees(np.arccos(c)))
    except Exception:
        return np.nan


def _extract_quality_safe_3d(out):
    """
    Extract frac_inliers and mean_error_um safely from a NucleiSky3D output dict.
    """
    if not isinstance(out, dict):
        return 0.0, np.nan

    q = out.get("match_quality", {}) or {}

    try:
        frac = float(q.get("frac_inliers", 0.0) or 0.0)
    except Exception:
        frac = 0.0

    try:
        err = q.get("mean_error_um", np.nan)
        err = float(err) if err is not None else np.nan
    except Exception:
        err = np.nan

    return frac, err


def _benchmark_runtime_overrides_3d(matcher, base_overrides, run_seed):
    """
    Prepare matcher runtime overrides.

    Accepts either:
      - hierarchical overrides: {"_common": {...}, "pyramid": {...}}
      - flat overrides: {"n_iters": 50000, ...}, interpreted as matcher-specific

    Always injects _common.random_state last for deterministic benchmark behaviour.
    """
    matcher_mode = _normalise_matcher_name_3d(matcher)

    out = {
        "_common": {},
        matcher_mode: {},
    }

    if base_overrides:
        if not isinstance(base_overrides, dict):
            raise ValueError("matcher_kwargs must be a dictionary.")

        is_hierarchical = ("_common" in base_overrides) or (matcher_mode in base_overrides)

        if is_hierarchical:
            out = _deep_merge_dict_3d(out, base_overrides)
        else:
            out[matcher_mode] = dict(base_overrides)

    out.setdefault("_common", {})
    out["_common"]["random_state"] = int(run_seed)

    return out


def _sanitize_controller_kwargs_3d(d):
    """
    Keep only kwargs supported by run_adaptive_nucleisky_3d.
    This prevents matcher-specific kwargs from being accidentally forwarded to
    the adaptive controller.
    """
    if d is None:
        return {}

    if not isinstance(d, dict):
        raise ValueError("For matcher='adaptive', matcher_kwargs must be a dict of controller options.")

    allowed = {
        "matcher_order",
        "base_seed",
        "matcher_config",
        "stop_on_success",
        "store_full_out",
        "max_total_time_s",
        "verbose",
        "return_dists",
        "print_fn",
    }

    out = {k: v for k, v in d.items() if k in allowed}
    dropped = [k for k in d.keys() if k not in allowed]

    if dropped:
        print(f"adaptive: dropped unsupported controller kwargs: {dropped}")

    return out


# ---------------------------------------------------------
# Candidate generation
# ---------------------------------------------------------

def make_patch_candidates_anchored_on_nuclei_3d(
    Z,
    Y,
    X,
    patch_shape_px_zyx,
    zs_px,
    ys_px,
    xs_px,
    max_trials,
    seed,
    min_nuclei=1,
    oversample_factor=25,
):
    """
    Deterministically propose up to max_trials patch origins (z0, y0, x0).

    Candidate patches are anchored on existing nuclei and filtered so that each
    patch contains at least min_nuclei landmarks. This mirrors the 2D benchmark
    logic and ensures informative query subvolumes.
    """
    pz, py, px = [int(v) for v in patch_shape_px_zyx]
    max_trials = int(max_trials)
    min_nuclei = int(min_nuclei)

    if Z <= pz or Y <= py or X <= px:
        return np.zeros((0, 3), dtype=int)

    zs_px = np.asarray(zs_px, dtype=int)
    ys_px = np.asarray(ys_px, dtype=int)
    xs_px = np.asarray(xs_px, dtype=int)

    if zs_px.size == 0:
        return np.zeros((0, 3), dtype=int)

    rng = np.random.default_rng(int(seed))

    n_proposals = max(max_trials * int(oversample_factor), max_trials)
    idx = rng.integers(0, len(zs_px), size=n_proposals, endpoint=False)

    out = []
    seen = set()

    for i in idx:
        z = int(zs_px[i])
        y = int(ys_px[i])
        x = int(xs_px[i])

        # Place the chosen nucleus at a random position inside the patch.
        oz = int(rng.integers(0, pz))
        oy = int(rng.integers(0, py))
        ox = int(rng.integers(0, px))

        z0, y0, x0 = _clip_origin_zyx(
            z - oz,
            y - oy,
            x - ox,
            Z,
            Y,
            X,
            pz,
            py,
            px,
        )

        key = (z0, y0, x0)

        if key in seen:
            continue

        if min_nuclei > 1:
            n_in = _count_nuclei_in_patch_3d(
                zs_px,
                ys_px,
                xs_px,
                z0,
                y0,
                x0,
                pz,
                py,
                px,
            )

            if n_in < min_nuclei:
                continue

        seen.add(key)
        out.append(key)

        if len(out) >= max_trials:
            break

    return np.asarray(out, dtype=int)


def make_benchmark_patch_plan_3d(
    vol_full_zyx,
    patch_shapes_px_zyx,
    patches_per_shape,
    seed=0,
    df_full=None,
    min_nuclei=1,
):
    """
    Build a fixed patch plan shared by all matchers.

    Returns
    -------
    dict
        Keys are patch shape tuples (pz, py, px), values are arrays of
        candidate origins with shape (N, 3), columns (z0, y0, x0).
    """
    vol_full_zyx = np.asarray(vol_full_zyx)

    if vol_full_zyx.ndim != 3:
        raise ValueError(f"Expected a 3D ZYX volume. Got shape={vol_full_zyx.shape}")

    Z, Y, X = vol_full_zyx.shape
    plan = {}

    if df_full is not None:
        needed = {"centroid_z_px", "centroid_y_px", "centroid_x_px"}

        if not needed.issubset(df_full.columns):
            raise ValueError(f"df_full must contain {sorted(needed)} to generate nucleus-anchored patches.")

        zs_px = df_full["centroid_z_px"].to_numpy(dtype=int, copy=False)
        ys_px = df_full["centroid_y_px"].to_numpy(dtype=int, copy=False)
        xs_px = df_full["centroid_x_px"].to_numpy(dtype=int, copy=False)
    else:
        zs_px = ys_px = xs_px = None

    for shape in patch_shapes_px_zyx:
        pz, py, px = [int(v) for v in shape]

        # Deterministic seed per patch shape.
        ss = np.random.SeedSequence([int(seed), int(pz), int(py), int(px), 3001])
        shape_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        if df_full is not None:
            candidates = make_patch_candidates_anchored_on_nuclei_3d(
                Z=Z,
                Y=Y,
                X=X,
                patch_shape_px_zyx=(pz, py, px),
                zs_px=zs_px,
                ys_px=ys_px,
                xs_px=xs_px,
                max_trials=int(patches_per_shape),
                seed=int(shape_seed),
                min_nuclei=int(min_nuclei),
            )
        else:
            rng = np.random.default_rng(int(shape_seed))

            if Z <= pz or Y <= py or X <= px:
                candidates = np.zeros((0, 3), dtype=int)
            else:
                z0 = rng.integers(0, Z - pz + 1, size=int(patches_per_shape))
                y0 = rng.integers(0, Y - py + 1, size=int(patches_per_shape))
                x0 = rng.integers(0, X - px + 1, size=int(patches_per_shape))
                candidates = np.column_stack([z0, y0, x0]).astype(int, copy=False)

        plan[(pz, py, px)] = candidates

    return plan


# ---------------------------------------------------------
# Data preparation helpers
# ---------------------------------------------------------

def _ensure_centroid_columns_3d(df_full, voxel_size_um_zyx):
    """
    Ensure df_full has both pixel and micrometre centroid columns.

    Required columns:
      centroid_z_px, centroid_y_px, centroid_x_px
      centroid_z_um, centroid_y_um, centroid_x_um
    """
    df = pd.DataFrame(df_full).copy()
    vz, vy, vx = _as_tuple3_3d(voxel_size_um_zyx, "voxel_size_um_zyx")

    px_cols = ["centroid_z_px", "centroid_y_px", "centroid_x_px"]
    um_cols = ["centroid_z_um", "centroid_y_um", "centroid_x_um"]

    has_px = all(c in df.columns for c in px_cols)
    has_um = all(c in df.columns for c in um_cols)

    if not has_px and not has_um:
        raise ValueError(
            "df_full must contain either pixel centroid columns "
            "(centroid_z_px, centroid_y_px, centroid_x_px) or micrometre centroid columns "
            "(centroid_z_um, centroid_y_um, centroid_x_um)."
        )

    if not has_px and has_um:
        df["centroid_z_px"] = df["centroid_z_um"].astype(float) / vz
        df["centroid_y_px"] = df["centroid_y_um"].astype(float) / vy
        df["centroid_x_px"] = df["centroid_x_um"].astype(float) / vx

    if not has_um and has_px:
        df["centroid_z_um"] = df["centroid_z_px"].astype(float) * vz
        df["centroid_y_um"] = df["centroid_y_px"].astype(float) * vy
        df["centroid_x_um"] = df["centroid_x_px"].astype(float) * vx

    return df


def _make_failure_row_3d(
    *,
    matcher,
    pz,
    py,
    px,
    z0,
    y0,
    x0,
    n_nuclei=0,
    failure_reason="failed",
    time_matcher_s=0.0,
    time_total_s=0.0,
):
    """
    Build a benchmark row for a failure or skipped patch.
    """
    row = {c: np.nan for c in BENCH_COLS_3D}

    row.update({
        "matcher": _normalise_matcher_name_3d(matcher),

        "patch_z_px": int(pz),
        "patch_y_px": int(py),
        "patch_x_px": int(px),
        "patch_shape_label": f"{int(pz)}x{int(py)}x{int(px)}",
        "patch_volume_px": int(pz) * int(py) * int(px),
        "patch_xy_area_px": int(py) * int(px),
        "patch_z0": int(z0),
        "patch_y0": int(y0),
        "patch_x0": int(x0),

        "n_nuclei": int(n_nuclei),
        "success": False,
        "frac_inliers": 0.0,
        "mean_error_um": np.nan,
        "best_scale": np.nan,
        "estimated_rotation_deg": np.nan,

        "z_error_um": np.nan,
        "y_error_um": np.nan,
        "x_error_um": np.nan,
        "trans_error_um": np.nan,
        "z_error_px": np.nan,
        "y_error_px": np.nan,
        "x_error_px": np.nan,
        "trans_error_px": np.nan,

        "ssim_xy": np.nan,

        "time_matcher_s": float(time_matcher_s),
        "time_total_s": float(time_total_s),

        "failure_reason": str(failure_reason),
    })

    return row


# ---------------------------------------------------------
# Single matcher benchmark
# ---------------------------------------------------------

def run_single_patch_match_benchmark_3d(
    vol_full_zyx,
    df_full,
    voxel_size_um_zyx,
    patch_shape_px_zyx,
    max_trials=30,
    random_state=0,
    compute_ssim=True,
    matcher="pyramid",

    matcher_config=None,
    matcher_kwargs=None,

    save_path=None,
    restart=False,
    candidates=None,
    min_nuclei_skip=3,
    ij_percentile_normalize=ij_percentile_normalize,
):
    """
    Benchmark one 3D matcher on a fixed set of 3D patch candidates.

    This function is designed to be called by
    benchmark_patch_shapes_on_current_volume_multi_3d(), which ensures that all
    matchers receive the same candidate patches for each patch size.
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize for SSIM calculation.")

    vol_full_zyx = np.asarray(vol_full_zyx)

    if vol_full_zyx.ndim != 3:
        raise ValueError(f"Expected a 3D ZYX volume. Got shape={vol_full_zyx.shape}")

    Z, Y, X = vol_full_zyx.shape
    pz, py, px = [int(v) for v in patch_shape_px_zyx]
    voxel_size_um_zyx = _as_tuple3_3d(voxel_size_um_zyx, "voxel_size_um_zyx")

    if Z <= pz or Y <= py or X <= px:
        print(f"Patch {patch_shape_px_zyx} is larger than volume {vol_full_zyx.shape}; skipping.")
        return []

    matcher_mode = _normalise_matcher_name_3d(matcher)

    df_full = _ensure_centroid_columns_3d(df_full, voxel_size_um_zyx)

    centroids_full_um = df_full[
        ["centroid_z_um", "centroid_y_um", "centroid_x_um"]
    ].to_numpy(dtype=float, copy=False)

    if candidates is None:
        candidates = make_patch_candidates_anchored_on_nuclei_3d(
            Z=Z,
            Y=Y,
            X=X,
            patch_shape_px_zyx=(pz, py, px),
            zs_px=df_full["centroid_z_px"].to_numpy(dtype=int, copy=False),
            ys_px=df_full["centroid_y_px"].to_numpy(dtype=int, copy=False),
            xs_px=df_full["centroid_x_px"].to_numpy(dtype=int, copy=False),
            max_trials=int(max_trials),
            seed=int(random_state),
            min_nuclei=int(min_nuclei_skip),
        )
    else:
        candidates = np.asarray(candidates, dtype=int)

    if candidates.ndim != 2 or candidates.shape[1] != 3:
        raise ValueError(f"candidates must be an array with shape (N, 3). Got {candidates.shape}")

    # -----------------------------------------------------
    # Checkpoint handling
    # -----------------------------------------------------

    results = []
    already_done = set()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        if restart and save_path.exists():
            save_path.unlink()
            print(f"Restart=True: deleted checkpoint {save_path}")

        if (not restart) and save_path.exists():
            try:
                prev = pd.read_csv(save_path)

                if {"patch_z0", "patch_y0", "patch_x0"}.issubset(prev.columns):
                    for _, r in prev.iterrows():
                        if pd.notna(r["patch_z0"]) and pd.notna(r["patch_y0"]) and pd.notna(r["patch_x0"]):
                            already_done.add((int(r["patch_z0"]), int(r["patch_y0"]), int(r["patch_x0"])))

                results = prev.to_dict(orient="records")

                print(
                    f"Resuming {matcher_mode} {pz}x{py}x{px} from checkpoint: "
                    f"{len(results)} rows loaded; {len(already_done)} completed coordinates."
                )

            except Exception as e:
                print(f"Could not read checkpoint {save_path}; starting fresh. Error: {e}")

    def _append_and_checkpoint(row):
        row = dict(row)

        for col in BENCH_COLS_3D:
            row.setdefault(col, np.nan)

        row = {col: row.get(col, np.nan) for col in BENCH_COLS_3D}
        results.append(row)

        if save_path is not None:
            write_header = not save_path.exists()
            pd.DataFrame([row], columns=BENCH_COLS_3D).to_csv(
                save_path,
                mode="a",
                header=write_header,
                index=False,
            )

        already_done.add((int(row["patch_z0"]), int(row["patch_y0"]), int(row["patch_x0"])))

    # -----------------------------------------------------
    # Main patch loop
    # -----------------------------------------------------

    run_seed = int(random_state if random_state is not None else 0)

    desc = f"{matcher_mode} | {pz}x{py}x{px}"

    for z0, y0, x0 in tqdm(candidates, total=len(candidates), desc=desc):
        z0 = int(z0)
        y0 = int(y0)
        x0 = int(x0)

        if (z0, y0, x0) in already_done:
            continue

        patch_start_t = time.perf_counter()

        z1 = z0 + pz
        y1 = y0 + py
        x1 = x0 + px

        if z1 > Z or y1 > Y or x1 > X:
            row = _make_failure_row_3d(
                matcher=matcher_mode,
                pz=pz,
                py=py,
                px=px,
                z0=z0,
                y0=y0,
                x0=x0,
                failure_reason="patch_out_of_bounds",
                time_total_s=time.perf_counter() - patch_start_t,
            )
            _append_and_checkpoint(row)
            continue

        patch_vol = vol_full_zyx[z0:z1, y0:y1, x0:x1]

        mask_patch = (
            (df_full["centroid_z_px"] >= z0) & (df_full["centroid_z_px"] < z1) &
            (df_full["centroid_y_px"] >= y0) & (df_full["centroid_y_px"] < y1) &
            (df_full["centroid_x_px"] >= x0) & (df_full["centroid_x_px"] < x1)
        )

        df_patch = df_full.loc[mask_patch].copy()
        n_nuc = int(len(df_patch))

        if n_nuc < int(min_nuclei_skip):
            row = _make_failure_row_3d(
                matcher=matcher_mode,
                pz=pz,
                py=py,
                px=px,
                z0=z0,
                y0=y0,
                x0=x0,
                n_nuclei=n_nuc,
                failure_reason="too_few_nuclei",
                time_total_s=time.perf_counter() - patch_start_t,
            )
            _append_and_checkpoint(row)
            continue

        vz, vy, vx = voxel_size_um_zyx
        true_origin_um = np.array([z0 * vz, y0 * vy, x0 * vx], dtype=float)

        centroids_crop_um = df_patch[
            ["centroid_z_um", "centroid_y_um", "centroid_x_um"]
        ].to_numpy(dtype=float, copy=False) - true_origin_um

        # Deterministic patch-specific seed.
        seed_parts = [
            int(run_seed),
            int(pz),
            int(py),
            int(px),
            int(z0),
            int(y0),
            int(x0),
            _stable_u32_from_str_3d(matcher_mode),
            9001,
        ]
        ss = np.random.SeedSequence(seed_parts)
        patch_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        # -------------------------------------------------
        # Run matcher
        # -------------------------------------------------

        t0_match = time.perf_counter()

        inputs = dict(
            centroids_crop_um=centroids_crop_um,
            centroids_full_um=centroids_full_um,
            full_shape_px_zyx=tuple(vol_full_zyx.shape),
            crop_shape_px_zyx=(int(pz), int(py), int(px)),
            pixel_size_full_um_zyx=voxel_size_um_zyx,
            pixel_size_crop_um_zyx=voxel_size_um_zyx,
            df_full=df_full,
            df_crop=df_patch,
        )

        out = {}

        try:
            if matcher_mode == "adaptive":
                controller_kwargs = _sanitize_controller_kwargs_3d(matcher_kwargs)
                controller_kwargs = dict(controller_kwargs)
                controller_kwargs.setdefault("base_seed", int(patch_seed))
                controller_kwargs.setdefault("stop_on_success", True)
                controller_kwargs.setdefault("store_full_out", False)

                if matcher_config is not None:
                    controller_kwargs.setdefault("matcher_config", matcher_config)

                out, _history = run_adaptive_nucleisky_3d(
                    **inputs,
                    **controller_kwargs,
                )

            else:
                runtime_overrides = _benchmark_runtime_overrides_3d(
                    matcher=matcher_mode,
                    base_overrides=matcher_kwargs,
                    run_seed=patch_seed,
                )

                out = NucleiSky3D(
                    matcher=matcher_mode,
                    matcher_config=matcher_config,
                    matcher_kwargs=runtime_overrides,
                    **inputs,
                )

        except Exception:
            traceback.print_exc()
            out = {}

        time_matcher_s = time.perf_counter() - t0_match

        # -------------------------------------------------
        # Extract outputs and diagnostics
        # -------------------------------------------------

        frac_inliers, mean_error_um = _extract_quality_safe_3d(out)

        if not isinstance(out, dict) or out.get("best_t", None) is None:
            row = _make_failure_row_3d(
                matcher=matcher_mode,
                pz=pz,
                py=py,
                px=px,
                z0=z0,
                y0=y0,
                x0=x0,
                n_nuclei=n_nuc,
                failure_reason="no_transform",
                time_matcher_s=time_matcher_s,
                time_total_s=time.perf_counter() - patch_start_t,
            )
            row["frac_inliers"] = float(frac_inliers)
            row["mean_error_um"] = mean_error_um
            _append_and_checkpoint(row)
            continue

        best_t = np.asarray(out["best_t"], dtype=float).reshape(3,)
        delta_um = best_t - true_origin_um

        z_error_um = float(delta_um[0])
        y_error_um = float(delta_um[1])
        x_error_um = float(delta_um[2])
        trans_error_um = float(np.linalg.norm(delta_um))

        delta_px = delta_um / np.asarray(voxel_size_um_zyx, dtype=float)

        z_error_px = float(delta_px[0])
        y_error_px = float(delta_px[1])
        x_error_px = float(delta_px[2])
        trans_error_px = float(np.linalg.norm(delta_px))

        best_scale = out.get("best_scale", np.nan)
        try:
            best_scale = float(best_scale)
        except Exception:
            best_scale = np.nan

        estimated_rotation_deg = _rotation_angle_from_identity_deg(out.get("best_R", None))

        ssim_xy = np.nan

        if compute_ssim:
            z_pred0 = int(round(best_t[0] / vz))
            y_pred0 = int(round(best_t[1] / vy))
            x_pred0 = int(round(best_t[2] / vx))

            if (
                0 <= z_pred0 <= Z - pz and
                0 <= y_pred0 <= Y - py and
                0 <= x_pred0 <= X - px
            ):
                pred_vol = vol_full_zyx[
                    z_pred0:z_pred0 + pz,
                    y_pred0:y_pred0 + py,
                    x_pred0:x_pred0 + px,
                ]

                try:
                    patch_mip = np.max(patch_vol, axis=0)
                    pred_mip = np.max(pred_vol, axis=0)

                    patch_norm = ij_percentile_normalize(patch_mip)
                    pred_norm = ij_percentile_normalize(pred_mip)

                    ssim_xy = float(ssim(patch_norm, pred_norm, data_range=1.0))

                except Exception:
                    ssim_xy = np.nan

        patch_total_t = time.perf_counter() - patch_start_t

        row = {
            "matcher": matcher_mode,

            "patch_z_px": int(pz),
            "patch_y_px": int(py),
            "patch_x_px": int(px),
            "patch_shape_label": f"{int(pz)}x{int(py)}x{int(px)}",
            "patch_volume_px": int(pz) * int(py) * int(px),
            "patch_xy_area_px": int(py) * int(px),
            "patch_z0": int(z0),
            "patch_y0": int(y0),
            "patch_x0": int(x0),

            "n_nuclei": int(n_nuc),

            "success": bool(out.get("success", False)),
            "frac_inliers": float(frac_inliers),
            "mean_error_um": mean_error_um,
            "best_scale": best_scale,
            "estimated_rotation_deg": estimated_rotation_deg,

            "z_error_um": z_error_um,
            "y_error_um": y_error_um,
            "x_error_um": x_error_um,
            "trans_error_um": trans_error_um,
            "z_error_px": z_error_px,
            "y_error_px": y_error_px,
            "x_error_px": x_error_px,
            "trans_error_px": trans_error_px,

            "ssim_xy": ssim_xy,

            "time_matcher_s": float(time_matcher_s),
            "time_total_s": float(patch_total_t),

            "failure_reason": "",
        }

        _append_and_checkpoint(row)

    return results


# ---------------------------------------------------------
# Multi-shape / multi-matcher benchmark
# ---------------------------------------------------------

def benchmark_patch_shapes_on_current_volume_multi_3d(
    vol_full_zyx,
    df_full,
    voxel_size_um_zyx,
    patch_shapes_px_zyx=((16, 128, 128), (32, 160, 160), (48, 192, 192)),
    patches_per_shape=30,
    random_state=0,
    matchers=("pyramid", "hashing", "adaptive"),

    matcher_config=None,
    matcher_kwargs_map=None,

    save_path_dir=None,
    restart=False,
    ij_percentile_normalize=ij_percentile_normalize,

    min_nuclei=3,
):
    """
    Run all selected 3D matchers on the same candidate subvolumes for each patch shape.

    This is the 3D equivalent of the 2D benchmark multi-run function.
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize.")

    if matcher_kwargs_map is None:
        matcher_kwargs_map = {}

    vol_full_zyx = np.asarray(vol_full_zyx)

    if vol_full_zyx.ndim != 3:
        raise ValueError(f"Expected a 3D ZYX volume. Got shape={vol_full_zyx.shape}")

    voxel_size_um_zyx = _as_tuple3_3d(voxel_size_um_zyx, "voxel_size_um_zyx")
    df_full = _ensure_centroid_columns_3d(df_full, voxel_size_um_zyx)

    patch_plan = make_benchmark_patch_plan_3d(
        vol_full_zyx=vol_full_zyx,
        patch_shapes_px_zyx=patch_shapes_px_zyx,
        patches_per_shape=int(patches_per_shape),
        seed=int(random_state),
        df_full=df_full,
        min_nuclei=int(max(1, min_nuclei)),
    )

    all_results = []

    for shape in patch_shapes_px_zyx:
        pz, py, px = [int(v) for v in shape]
        candidates = patch_plan.get((pz, py, px), np.zeros((0, 3), dtype=int))

        print(f"\n=== 3D patch shape {pz}×{py}×{px} px | N candidates = {len(candidates)} ===")

        if len(candidates) == 0:
            print("No valid candidate patches for this shape. Skipping.")
            continue

        for matcher in matchers:
            matcher_mode = _normalise_matcher_name_3d(matcher)

            print(f"\n========== 3D matcher: {matcher_mode} ==========")

            checkpoint_path = None

            if save_path_dir is not None:
                save_dir = Path(save_path_dir)
                save_dir.mkdir(parents=True, exist_ok=True)
                checkpoint_path = save_dir / f"{matcher_mode}_patch{pz}x{py}x{px}.csv"

            # Deterministic per global seed, patch shape and matcher.
            ss_run = np.random.SeedSequence([
                int(random_state),
                int(pz),
                int(py),
                int(px),
                _stable_u32_from_str_3d(matcher_mode),
                7001,
            ])
            run_seed = int(ss_run.generate_state(1, dtype=np.uint32)[0])

            rows = run_single_patch_match_benchmark_3d(
                vol_full_zyx=vol_full_zyx,
                df_full=df_full,
                voxel_size_um_zyx=voxel_size_um_zyx,
                patch_shape_px_zyx=(pz, py, px),
                max_trials=int(patches_per_shape),
                random_state=int(run_seed),
                compute_ssim=True,
                matcher=matcher_mode,

                matcher_config=matcher_config,
                matcher_kwargs=matcher_kwargs_map.get(matcher_mode, None),

                save_path=checkpoint_path,
                restart=bool(restart),
                candidates=candidates,
                min_nuclei_skip=int(min_nuclei),

                ij_percentile_normalize=ij_percentile_normalize,
            )

            all_results.extend(rows)

    df_results_3d = pd.DataFrame(all_results)

    # Ensure stable column order.
    for col in BENCH_COLS_3D:
        if col not in df_results_3d.columns:
            df_results_3d[col] = np.nan

    df_results_3d = df_results_3d[BENCH_COLS_3D]

    return all_results, df_results_3d


print("✅ 3D benchmark functions are ready.")

# 1. Load your volume and extract features

In [ ]:
#@title Step 1: Load reference 3D volume + voxel size

import json
import traceback
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------------------------------------------------
# Optional imports / checks
# ---------------------------------------------------------------------

# This cell expects the NucleiSky3D app/import cells to have defined:
#   inspect_volume_header
#   load_volume
#   require_voxel_size_um_zyx
#
# It also optionally uses:
#   imshow_safe3d
#   make_result_dir
#   STYLE_CSS
#   _status_ok / _status_err / _status_info
#
# If status / display helpers are missing, lightweight fallbacks are provided.

required_app_functions = [
    "inspect_volume_header",
    "load_volume",
    "require_voxel_size_um_zyx",
]

missing_app_functions = [fn for fn in required_app_functions if fn not in globals()]
if missing_app_functions:
    raise RuntimeError(
        "Missing required NucleiSky3D loading functions: "
        + ", ".join(missing_app_functions)
        + ". Run the NucleiSky3D setup/import cells first."
    )


def _fallback_status_info(lines, title="Info"):
    if isinstance(lines, str):
        lines = [lines]
    body = "<br>".join(map(str, lines))
    return f"<div style='padding:10px;border-left:4px solid #3B82F6;background:#EFF6FF;'><b>{title}</b><br>{body}</div>"


def _fallback_status_ok(lines, title="Success"):
    if isinstance(lines, str):
        lines = [lines]
    body = "<br>".join(map(str, lines))
    return f"<div style='padding:10px;border-left:4px solid #16A34A;background:#F0FDF4;'><b>{title}</b><br>{body}</div>"


def _fallback_status_err(lines, title="Error"):
    if isinstance(lines, str):
        lines = [lines]
    body = "<br>".join(map(str, lines))
    return f"<div style='padding:10px;border-left:4px solid #DC2626;background:#FEF2F2;'><b>{title}</b><br>{body}</div>"


_status_info_fn = globals().get("_status_info", _fallback_status_info)
_status_ok_fn = globals().get("_status_ok", _fallback_status_ok)
_status_err_fn = globals().get("_status_err", _fallback_status_err)


def _imshow_safe(ax, img, title=""):
    """
    Use app helper if available; otherwise use a simple robust imshow.
    """
    if "imshow_safe3d" in globals():
        imshow_safe3d(ax, img, title=title)
        return

    arr = np.asarray(img)
    lo, hi = np.nanpercentile(arr, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = np.nanmin(arr), np.nanmax(arr)

    ax.imshow(arr, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(title)
    ax.axis("off")


def _mip_xy(vol_zyx):
    return np.max(np.asarray(vol_zyx), axis=0)


def _mip_xz(vol_zyx):
    return np.max(np.asarray(vol_zyx), axis=1)


def _mip_yz(vol_zyx):
    return np.max(np.asarray(vol_zyx), axis=2)


def _parse_voxel_size_zyx(text):
    """
    Parse optional manual voxel size as z,y,x in micrometres.
    Empty string means: infer from metadata.
    """
    text = (text or "").strip()
    if not text:
        return None

    parts = [p.strip() for p in text.split(",") if p.strip()]
    if len(parts) != 3:
        raise ValueError("Voxel size must be three comma-separated values in Z,Y,X order, for example: 0.3,0.1,0.1")

    values = tuple(float(p) for p in parts)
    if any(v <= 0 for v in values):
        raise ValueError("Voxel-size values must be positive.")

    return values


def _shape_ndim(header):
    shape = tuple(header.get("shape") or ())
    return shape, len(shape)


def _configure_channel_controls(header, axis_widget, index_widget, box_widget):
    """
    If the input is 4D, expose channel-axis and channel-index selection.
    If the input is already 3D, hide channel selection.
    """
    shape, ndim = _shape_ndim(header)

    if ndim == 4:
        axis_opts = [(f"axis {ax} (size={shape[ax]})", ax) for ax in range(4)]
        axis_widget.options = [("Select axis", None)] + axis_opts
        axis_widget.value = None
        index_widget.options = [("Select index", None)]
        index_widget.value = None
        box_widget.layout.display = "flex"
    else:
        axis_widget.options = [("N/A for 3D", None)]
        axis_widget.value = None
        index_widget.options = [("N/A for 3D", None)]
        index_widget.value = None
        box_widget.layout.display = "none"


def _refresh_channel_index(axis_widget, index_widget, header):
    shape, ndim = _shape_ndim(header)

    if ndim != 4 or axis_widget.value is None:
        index_widget.options = [("Select index", None)]
        index_widget.value = None
        return

    axis = int(axis_widget.value)
    axis_size = int(shape[axis])

    index_widget.options = [("Select index", None)] + [(str(i), i) for i in range(axis_size)]
    index_widget.value = None


def _resolve_channel_selection(header, axis_widget, index_widget):
    """
    Return channel_axis, channel_index for load_volume.
    For 3D input, return None, 0.
    """
    shape, ndim = _shape_ndim(header)

    if ndim == 3:
        return None, 0

    if ndim != 4:
        raise ValueError(f"Reference volume must be 3D or 4D. Got shape={shape}")

    if axis_widget.value is None or index_widget.value is None:
        raise ValueError("The reference volume is 4D. Please select the channel axis and channel index.")

    axis = int(axis_widget.value)
    index = int(index_widget.value)

    if index < 0 or index >= int(shape[axis]):
        raise ValueError(f"Channel index {index} is out of bounds for axis {axis} with size {shape[axis]}.")

    return axis, index


def _make_default_result_dir(full_path_str):
    """
    Reuse app make_result_dir if available, otherwise create a simple folder.
    """
    if "make_result_dir" in globals():
        return Path(make_result_dir(big_image_path=full_path_str, tag="NucleiSky3D"))

    stem = Path(full_path_str).stem or "reference_volume"
    return Path(f"nucleisky3d_results_{stem}")


# ---------------------------------------------------------------------
# User-facing defaults
# ---------------------------------------------------------------------

FULL_VOLUME_PATH = globals().get("FULL_VOLUME_PATH", "")
VOXEL_SIZE_UM_ZYX = globals().get("VOXEL_SIZE_UM_ZYX", None)
RESULT_DIR = globals().get("RESULT_DIR", "nucleisky3d_results")

_vox_default = "" if VOXEL_SIZE_UM_ZYX is None else ",".join(map(str, VOXEL_SIZE_UM_ZYX))

# ---------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------

style_html = widgets.HTML(globals().get("STYLE_CSS", ""))

title_html = widgets.HTML(
    "<div style='font-size:22px;font-weight:700;margin-bottom:4px;'>Load reference 3D volume</div>"
    "<div style='color:#64748B;margin-bottom:12px;'>Load the full/reference volume, infer or enter voxel size, and display orthogonal maximum-intensity projections.</div>"
)

path_w = widgets.Text(
    value=str(FULL_VOLUME_PATH),
    description="Full path",
    placeholder="/path/to/reference_volume.tif",
    layout=widgets.Layout(width="98%"),
)

channel_axis_w = widgets.Dropdown(
    options=[("N/A for 3D", None)],
    value=None,
    description="Channel axis",
    layout=widgets.Layout(width="260px"),
)

channel_index_w = widgets.Dropdown(
    options=[("N/A for 3D", None)],
    value=None,
    description="Channel index",
    layout=widgets.Layout(width="260px"),
)

channel_box = widgets.HBox(
    [channel_axis_w, channel_index_w],
    layout=widgets.Layout(display="none", gap="10px"),
)

vox_w = widgets.Text(
    value=_vox_default,
    description="Voxel Z,Y,X",
    placeholder="optional; e.g. 0.3,0.1,0.1",
    layout=widgets.Layout(width="98%"),
)

outdir_w = widgets.Text(
    value=str(RESULT_DIR),
    description="Result dir",
    placeholder="leave empty for auto-generated folder",
    layout=widgets.Layout(width="98%"),
)

inspect_btn = widgets.Button(
    description="Inspect header",
    button_style="info",
    icon="search",
    layout=widgets.Layout(width="180px"),
)

load_btn = widgets.Button(
    description="Load reference volume",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="240px"),
)

status_html = widgets.HTML("")
header_out = widgets.Output()
fig_out = widgets.Output()

for w in [path_w, vox_w, outdir_w, channel_axis_w, channel_index_w]:
    try:
        w.style.description_width = "100px"
    except Exception:
        pass

state = {
    "header": None,
    "last_path": "",
}


# ---------------------------------------------------------------------
# UI logic
# ---------------------------------------------------------------------

def inspect_reference_header(show_output=True):
    """
    Inspect the reference header and configure channel widgets.
    """
    path_str = path_w.value.strip()

    if not path_str:
        raise FileNotFoundError("Please provide a reference volume path.")

    if not Path(path_str).exists():
        raise FileNotFoundError(f"Reference volume not found: {path_str}")

    header = inspect_volume_header(path_str)
    shape, ndim = _shape_ndim(header)

    if ndim not in (3, 4):
        raise ValueError(f"Reference volume must be 3D or 4D. Got shape={shape}")

    state["header"] = header
    state["last_path"] = path_str

    _configure_channel_controls(header, channel_axis_w, channel_index_w, channel_box)

    if show_output:
        with header_out:
            clear_output()
            print("Header inspection")
            print("-----------------")
            print(header)
            print(f"\nDetected shape: {shape}")
            print(f"Detected ndim:  {ndim}")
            if ndim == 4:
                print("\n4D volume detected. Please select channel axis and index before loading.")
            else:
                print("\n3D volume detected. No channel selection needed.")

        status_html.value = _status_info_fn(
            [
                "Header inspected successfully.",
                f"Shape: {shape}",
                "Select a channel if the volume is 4D, then load the reference volume.",
            ],
            title="Header inspection",
        )

    return header


def on_channel_axis_change(_=None):
    header = state.get("header")
    if header is not None:
        _refresh_channel_index(channel_axis_w, channel_index_w, header)


def on_path_change(_=None):
    """
    Reset header/channel state when the path changes.
    """
    current = path_w.value.strip()
    if current != state.get("last_path", ""):
        state["header"] = None
        channel_box.layout.display = "none"
        channel_axis_w.options = [("N/A for 3D", None)]
        channel_axis_w.value = None
        channel_index_w.options = [("N/A for 3D", None)]
        channel_index_w.value = None


def on_inspect_clicked(_):
    try:
        inspect_reference_header(show_output=True)
    except Exception as e:
        status_html.value = _status_err_fn([str(e)], title="Header inspection failed")
        with header_out:
            clear_output()
            traceback.print_exc()


def on_load_clicked(_):
    """
    Load the reference volume only.
    Sets these globals:
        img_full
        voxel_size_full_um
        RESULT_DIR
        FULL_PATH_USED
        FULL_HEADER
        FULL_CHANNEL_AXIS
        FULL_CHANNEL_INDEX
    """
    global img_full, voxel_size_full_um, RESULT_DIR, FULL_PATH_USED
    global FULL_HEADER, FULL_CHANNEL_AXIS, FULL_CHANNEL_INDEX
    global FULL_VOLUME_PATH, VOXEL_SIZE_UM_ZYX

    status_html.value = _status_info_fn("Loading reference volume.", title="Processing")

    with fig_out:
        clear_output(wait=True)

    try:
        full_path_str = path_w.value.strip()
        if not full_path_str:
            raise FileNotFoundError("Please provide a reference volume path.")

        if not Path(full_path_str).exists():
            raise FileNotFoundError(f"Reference volume not found: {full_path_str}")

        # Inspect or reuse current header.
        header = state.get("header")
        if header is None or state.get("last_path") != full_path_str:
            header = inspect_reference_header(show_output=False)

        channel_axis, channel_index = _resolve_channel_selection(
            header,
            channel_axis_w,
            channel_index_w,
        )

        # Load the volume using the same app-level loader.
        img_full_load = load_volume(
            full_path_str,
            channel_axis=channel_axis,
            channel_index=channel_index,
        )

        img_full_load = np.asarray(img_full_load)

        if img_full_load.ndim != 3:
            raise ValueError(
                "Reference volume did not resolve to a 3D ZYX array. "
                f"Got shape={img_full_load.shape}."
            )

        # Voxel size: manual override if provided, otherwise metadata.
        manual_voxel = _parse_voxel_size_zyx(vox_w.value)

        if manual_voxel is not None:
            voxel_size = tuple(float(v) for v in manual_voxel)
            voxel_source = "manual"
        else:
            voxel_size = tuple(float(v) for v in require_voxel_size_um_zyx(full_path_str))
            voxel_source = "metadata"

        # Output directory
        outdir_text = outdir_w.value.strip()
        if outdir_text:
            result_dir = Path(outdir_text)
        else:
            result_dir = _make_default_result_dir(full_path_str)

        result_dir.mkdir(parents=True, exist_ok=True)

        # Set notebook globals for downstream 3D benchmark cells.
        img_full = img_full_load
        voxel_size_full_um = voxel_size
        RESULT_DIR = str(result_dir)
        FULL_PATH_USED = full_path_str
        FULL_HEADER = header
        FULL_CHANNEL_AXIS = channel_axis
        FULL_CHANNEL_INDEX = channel_index
        FULL_VOLUME_PATH = full_path_str
        VOXEL_SIZE_UM_ZYX = voxel_size

        # Save metadata for reproducibility.
        metadata = {
            "full_image_path": FULL_PATH_USED,
            "full_shape_zyx": list(img_full.shape),
            "voxel_size_full_um_zyx": list(voxel_size_full_um),
            "voxel_size_source": voxel_source,
            "full_axes": "ZYX",
            "full_channel_axis": FULL_CHANNEL_AXIS,
            "full_channel_index": FULL_CHANNEL_INDEX,
            "header": FULL_HEADER,
        }

        with open(result_dir / "inputs_metadata.json", "w") as f:
            json.dump(metadata, f, indent=2, default=str)

        status_html.value = _status_ok_fn(
            [
                "Reference volume loaded successfully.",
                f"Path: {FULL_PATH_USED}",
                f"Shape ZYX: {tuple(img_full.shape)}",
                f"Voxel size Z,Y,X: {voxel_size_full_um} µm ({voxel_source})",
                f"Result directory: {RESULT_DIR}",
                "Available globals: img_full, voxel_size_full_um, RESULT_DIR, FULL_PATH_USED",
            ],
            title="Success",
        )

        # Display orthogonal MIPs.
        with fig_out:
            fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=120)

            _imshow_safe(axes[0], _mip_xy(img_full), title="Reference MIP XY")
            _imshow_safe(axes[1], _mip_xz(img_full), title="Reference MIP XZ")
            _imshow_safe(axes[2], _mip_yz(img_full), title="Reference MIP YZ")

            fig.suptitle(
                f"Reference volume | shape ZYX={tuple(img_full.shape)} | voxel ZYX={voxel_size_full_um} µm",
                y=1.03,
                fontsize=11,
            )

            plt.tight_layout()
            plt.show()

    except Exception as e:
        status_html.value = _status_err_fn([str(e)], title="Error during reference loading")
        with fig_out:
            clear_output()
            traceback.print_exc()


channel_axis_w.observe(on_channel_axis_change, names="value")
path_w.observe(on_path_change, names="value")
inspect_btn.on_click(on_inspect_clicked)
load_btn.on_click(on_load_clicked)

# ---------------------------------------------------------------------
# Display UI
# ---------------------------------------------------------------------

ui = widgets.VBox(
    [
        style_html,
        title_html,
        widgets.HTML("<b>Reference volume</b>"),
        path_w,
        channel_box,
        widgets.HTML("<b>Voxel size</b><br><span style='color:#64748B;'>Leave empty to read from metadata. Enter Z,Y,X in µm to override or provide missing metadata.</span>"),
        vox_w,
        widgets.HTML("<b>Output</b>"),
        outdir_w,
        widgets.HBox([inspect_btn, load_btn], layout=widgets.Layout(gap="10px")),
        widgets.Box([status_html], layout=widgets.Layout(width="100%", margin="10px 0")),
        header_out,
        fig_out,
    ],
    layout=widgets.Layout(width="95%"),
)

display(ui)

In [ ]:
#@title Step 2: Reference-only 2.5D segmentation + 3D feature extraction

import json
import html
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------------------------------------------------
# Required functions from NucleiSky3D app/import cells
# ---------------------------------------------------------------------

required_functions = [
    "segment_nuclei_2p5d",
    "extract_nuclear_features_3d",
]

missing = [fn for fn in required_functions if fn not in globals()]
if missing:
    raise RuntimeError(
        "Missing required NucleiSky3D functions: "
        + ", ".join(missing)
        + ". Run the NucleiSky3D setup/import cells first."
    )

# Optional helpers from the app
STYLE_CSS_SAFE = globals().get("STYLE_CSS", "")

def _fallback_status_message(lines, title="Status", kind="info"):
    if isinstance(lines, str):
        lines = [lines]

    safe_lines = "<br>".join(html.escape(str(x)) for x in lines)

    colors = {
        "ok": ("#16A34A", "#F0FDF4"),
        "error": ("#DC2626", "#FEF2F2"),
        "warn": ("#D97706", "#FFFBEB"),
        "info": ("#2563EB", "#EFF6FF"),
    }

    border, bg = colors.get(kind, colors["info"])

    return (
        f"<div style='padding:10px;border-left:4px solid {border};"
        f"background:{bg};margin:8px 0;'>"
        f"<b>{html.escape(str(title))}</b><br>{safe_lines}</div>"
    )

_status_ok_fn = globals().get("_status_ok", lambda lines, title="Ready": _fallback_status_message(lines, title, "ok"))
_status_err_fn = globals().get("_status_err", lambda lines, title="Action required": _fallback_status_message(lines, title, "error"))
_status_info_fn = globals().get("_status_info", lambda lines, title="Working": _fallback_status_message(lines, title, "info"))


def _imshow_safe(ax, img, title="", max_dim=1500):
    """
    Use app display helper if available; otherwise use a robust grayscale imshow.
    """
    if "imshow_safe3d" in globals():
        out = imshow_safe3d(ax, img, title=title, max_dim=max_dim)
        return out

    arr = np.asarray(img)

    # Downsample for display only if very large.
    scale = 1.0
    if max(arr.shape) > max_dim:
        scale = max_dim / max(arr.shape)
        step = max(1, int(round(1 / scale)))
        arr_disp = arr[::step, ::step]
    else:
        arr_disp = arr

    lo, hi = np.nanpercentile(arr_disp, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = np.nanmin(arr_disp), np.nanmax(arr_disp)

    ax.imshow(arr_disp, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(title)
    ax.axis("off")
    return arr_disp


def _save_tiff_zyx(path, arr, voxel_size_um_zyx=None):
    """
    Use app save helper if available; otherwise fall back to tifffile.
    """
    path = Path(path)

    if "save_tiff_zyx" in globals():
        save_tiff_zyx(path, arr, voxel_size_um_zyx=voxel_size_um_zyx, compress="zlib")
        return

    import tifffile
    tifffile.imwrite(path, np.asarray(arr), compression="zlib")


def _load_label_volume(path):
    """
    Load a label volume using app load_volume if available; otherwise tifffile.
    """
    path = str(path)

    if "load_volume" in globals():
        return np.asarray(load_volume(path))

    import tifffile
    return np.asarray(tifffile.imread(path))


def _as_tuple3(v, name="voxel size"):
    if v is None:
        raise ValueError(f"{name} is missing.")

    vals = tuple(float(x) for x in v)
    if len(vals) != 3:
        raise ValueError(f"{name} must contain exactly three values in Z,Y,X order.")

    if any(x <= 0 for x in vals):
        raise ValueError(f"{name} values must be positive.")

    return vals


def _mip_xy(vol_zyx):
    return np.max(np.asarray(vol_zyx), axis=0)


def _mip_xz(vol_zyx):
    return np.max(np.asarray(vol_zyx), axis=1)


def _mip_yz(vol_zyx):
    return np.max(np.asarray(vol_zyx), axis=2)


def _best_z_slice_from_df(df, img_shape):
    """
    Choose a representative z-slice containing many detected nuclei.
    """
    if df is not None and len(df) > 0 and "centroid_z_px" in df.columns:
        z = int(df["centroid_z_px"].round().value_counts().idxmax())
    else:
        z = int(img_shape[0] // 2)

    return int(np.clip(z, 0, img_shape[0] - 1))


def _plot_reference_segmentation_overview(img_vol, labels_vol, df, title_prefix="Reference"):
    """
    Display MIPs and one representative segmentation overlay.
    """
    img_vol = np.asarray(img_vol)
    labels_vol = np.asarray(labels_vol)

    best_z = _best_z_slice_from_df(df, img_vol.shape)

    fig, axes = plt.subplots(2, 3, figsize=(13, 8), dpi=115)

    _imshow_safe(axes[0, 0], _mip_xy(img_vol), title=f"{title_prefix} raw MIP XY")
    _imshow_safe(axes[0, 1], _mip_xz(img_vol), title=f"{title_prefix} raw MIP XZ")
    _imshow_safe(axes[0, 2], _mip_yz(img_vol), title=f"{title_prefix} raw MIP YZ")

    raw_slice = img_vol[best_z]
    label_slice = labels_vol[best_z] if labels_vol.ndim == 3 else None

    disp_raw = _imshow_safe(
        axes[1, 0],
        raw_slice,
        title=f"{title_prefix} raw slice Z={best_z}",
    )

    _imshow_safe(
        axes[1, 1],
        raw_slice,
        title=f"{title_prefix} centroids Z={best_z}",
    )

    if df is not None and len(df) > 0 and "centroid_z_px" in df.columns:
        z_mask = np.abs(df["centroid_z_px"] - best_z) < 1.5
        df_slice = df.loc[z_mask].copy()

        if not df_slice.empty:
            scale_y = disp_raw.shape[0] / max(1, raw_slice.shape[0])
            scale_x = disp_raw.shape[1] / max(1, raw_slice.shape[1])

            axes[1, 1].scatter(
                df_slice["centroid_x_px"].to_numpy(dtype=float) * scale_x,
                df_slice["centroid_y_px"].to_numpy(dtype=float) * scale_y,
                s=14,
                c="red",
                alpha=0.8,
                edgecolors="white",
                linewidths=0.4,
            )

    if label_slice is not None:
        # Show labels as a quick QC image; binary display keeps it fast and readable.
        axes[1, 2].imshow(label_slice > 0, cmap="gray")
        axes[1, 2].set_title(f"{title_prefix} labels Z={best_z}")
        axes[1, 2].axis("off")
    else:
        axes[1, 2].axis("off")

    for ax in axes.ravel():
        ax.axis("off")

    fig.tight_layout()
    plt.show()


# ---------------------------------------------------------------------
# Pre-flight checks from Step 1
# ---------------------------------------------------------------------

if "img_full" not in globals():
    raise RuntimeError("img_full not found. Run Step 1 to load the reference volume first.")

if "voxel_size_full_um" not in globals():
    raise RuntimeError("voxel_size_full_um not found. Run Step 1 to load voxel size first.")

img_full = np.asarray(img_full)

if img_full.ndim != 3:
    raise ValueError(f"img_full must be a 3D ZYX volume. Got shape={img_full.shape}")

voxel_size_full_um = _as_tuple3(voxel_size_full_um, "voxel_size_full_um")

RESULT_DIR_LOCAL = Path(globals().get("RESULT_DIR", "./NucleiSky3D_Results"))
RESULT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------

style_html = widgets.HTML(STYLE_CSS_SAFE)

title_html = widgets.HTML(
    "<div style='font-size:22px;font-weight:700;margin-bottom:4px;'>Reference segmentation and 3D feature extraction</div>"
    "<div style='color:#64748B;margin-bottom:12px;'>Segment only the reference/full volume, or load an existing 3D label mask, then extract 3D centroid features for benchmarking.</div>"
)

source_mode = widgets.ToggleButtons(
    options=[
        ("Run 2.5D segmentation", "new"),
        ("Load existing labels", "existing"),
    ],
    value="new",
    style={"button_width": "190px"},
)

labels_path_w = widgets.Text(
    value="",
    description="Labels path",
    placeholder="/path/to/reference_labels.tif",
    layout=widgets.Layout(width="98%"),
)

normalize_before_w = widgets.Checkbox(
    value=False,
    description="Percentile normalize before segmentation",
    indent=False,
)

seg_method_w = widgets.Dropdown(
    options=[
        ("InstanSeg", "instanseg"),
        ("Cellpose", "cellpose"),
        ("Auto-threshold", "threshold"),
    ],
    value="instanseg",
    description="Method",
    layout=widgets.Layout(width="320px"),
)

# InstanSeg settings
inst_model_w = widgets.Dropdown(
    options=["brightfield_nuclei", "fluorescence_nuclei_and_cells"],
    value="brightfield_nuclei",
    description="Model",
    layout=widgets.Layout(width="360px"),
)

inst_target_w = widgets.Dropdown(
    options=["nuclei", "cells"],
    value="nuclei",
    description="Target",
    layout=widgets.Layout(width="240px"),
)

inst_mode_w = widgets.Dropdown(
    options=[
        ("Auto", "auto"),
        ("Small image", "small"),
        ("Medium / tiled", "medium"),
    ],
    value="auto",
    description="Mode",
    layout=widgets.Layout(width="260px"),
)

inst_clean_w = widgets.Checkbox(
    value=True,
    description="Cleanup fragments",
    indent=False,
)

inst_verbosity_w = widgets.IntSlider(
    value=0,
    min=0,
    max=2,
    step=1,
    description="Verbosity",
    layout=widgets.Layout(width="320px"),
)

inst_panel = widgets.VBox([
    widgets.HTML("<b>InstanSeg settings</b>"),
    widgets.HBox([inst_model_w, inst_target_w], layout=widgets.Layout(gap="10px")),
    widgets.HBox([inst_mode_w, inst_clean_w], layout=widgets.Layout(gap="10px")),
    inst_verbosity_w,
])

# Threshold settings
thr_method_w = widgets.Dropdown(
    options=["otsu", "li", "yen", "triangle", "isodata"],
    value="otsu",
    description="Algorithm",
    layout=widgets.Layout(width="280px"),
)

thr_sigma_w = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=5.0,
    step=0.25,
    description="Blur sigma",
    layout=widgets.Layout(width="360px"),
)

thr_min_obj_w = widgets.IntText(
    value=80,
    description="Min area",
    layout=widgets.Layout(width="220px"),
)

thr_watershed_w = widgets.Checkbox(
    value=True,
    description="Watershed split",
    indent=False,
)

thr_panel = widgets.VBox([
    widgets.HTML("<b>Threshold settings</b>"),
    widgets.HBox([thr_method_w, thr_min_obj_w, thr_watershed_w], layout=widgets.Layout(gap="10px")),
    thr_sigma_w,
])

stitch_iou_w = widgets.FloatSlider(
    value=0.30,
    min=0.01,
    max=0.99,
    step=0.05,
    description="Z-stitch IoU",
    layout=widgets.Layout(width="400px"),
)

k_neighbors_w = widgets.IntText(
    value=5,
    description="k-neighbors",
    layout=widgets.Layout(width="220px"),
)

run_btn = widgets.Button(
    description="Run reference segmentation + extract features",
    button_style="success",
    icon="cogs",
    layout=widgets.Layout(width="360px", height="40px"),
)

status_html = widgets.HTML("")
out_log = widgets.Output(
    layout=widgets.Layout(
        max_height="260px",
        overflow="auto",
        border="1px solid #E2E8F0",
        padding="8px",
    )
)
out_fig = widgets.Output()
out_tbl = widgets.Output(layout=widgets.Layout(max_height="300px", overflow="auto"))

for w in [
    labels_path_w,
    seg_method_w,
    inst_model_w,
    inst_target_w,
    inst_mode_w,
    inst_verbosity_w,
    thr_method_w,
    thr_sigma_w,
    thr_min_obj_w,
    stitch_iou_w,
    k_neighbors_w,
]:
    try:
        w.style.description_width = "100px"
    except Exception:
        pass


# ---------------------------------------------------------------------
# UI logic
# ---------------------------------------------------------------------

def _build_segmentation_settings():
    method = seg_method_w.value

    settings = {"method": method}

    if method == "instanseg":
        settings["instanseg"] = {
            "model_name": inst_model_w.value,
            "target": inst_target_w.value,
            "mode": inst_mode_w.value,
            "verbosity": int(inst_verbosity_w.value),
            "cleanup_fragments": bool(inst_clean_w.value),
        }

    elif method == "threshold":
        settings["threshold"] = {
            "method": thr_method_w.value,
            "gaussian_sigma_zyx": (0.0, float(thr_sigma_w.value), float(thr_sigma_w.value)),
            "min_object_size_vox": int(thr_min_obj_w.value),
            "do_watershed": bool(thr_watershed_w.value),
        }

    elif method == "cellpose":
        # Use defaults unless you add additional Cellpose-specific widgets later.
        settings["cellpose"] = {}

    else:
        raise ValueError(f"Unknown segmentation method: {method}")

    return settings


def _update_ui(_=None):
    use_existing = source_mode.value == "existing"

    labels_path_w.layout.display = "flex" if use_existing else "none"

    normalize_before_w.layout.display = "none" if use_existing else "flex"
    seg_method_w.layout.display = "none" if use_existing else "flex"
    stitch_iou_w.layout.display = "none" if use_existing else "flex"

    inst_panel.layout.display = "flex" if (not use_existing and seg_method_w.value == "instanseg") else "none"
    thr_panel.layout.display = "flex" if (not use_existing and seg_method_w.value == "threshold") else "none"

    if use_existing:
        run_btn.description = "Load reference labels + extract features"
    else:
        run_btn.description = "Run reference segmentation + extract features"


source_mode.observe(_update_ui, names="value")
seg_method_w.observe(_update_ui, names="value")
_update_ui()


# ---------------------------------------------------------------------
# Main handler
# ---------------------------------------------------------------------

def on_run_reference_segmentation(_):
    """
    Sets these globals:
        img_full_seg
        labels_full_seg
        voxel_full_seg
        df_full
        SEG_DIR
        LABELS_FULL_PATH
        FEATURES_FULL_PATH
    """
    global img_full_seg, labels_full_seg, voxel_full_seg, df_full
    global SEG_DIR, LABELS_FULL_PATH, FEATURES_FULL_PATH

    status_html.value = _status_info_fn("Running reference segmentation and feature extraction.", title="Processing")

    out_log.clear_output(wait=True)
    out_fig.clear_output(wait=True)
    out_tbl.clear_output(wait=True)

    try:
        result_dir = Path(globals().get("RESULT_DIR", RESULT_DIR_LOCAL))
        result_dir.mkdir(parents=True, exist_ok=True)

        SEG_DIR = result_dir / "segmentation"
        SEG_DIR.mkdir(parents=True, exist_ok=True)

        img_reference = np.asarray(globals()["img_full"])
        voxel_reference = _as_tuple3(globals()["voxel_size_full_um"], "voxel_size_full_um")

        if img_reference.ndim != 3:
            raise ValueError(f"img_full must be a 3D ZYX array. Got shape={img_reference.shape}")

        with out_log:
            print("Reference-only 3D segmentation workflow")
            print("---------------------------------------")
            print(f"Reference shape ZYX: {img_reference.shape}")
            print(f"Reference dtype: {img_reference.dtype}")
            print(f"Voxel size Z,Y,X: {voxel_reference} µm")
            print(f"Output directory: {SEG_DIR}")

            if source_mode.value == "existing":
                label_path = labels_path_w.value.strip()

                if not label_path:
                    raise FileNotFoundError("Please provide a path to an existing reference label volume.")

                if not Path(label_path).exists():
                    raise FileNotFoundError(f"Reference label volume not found: {label_path}")

                print("\nLoading existing reference labels...")
                labels_local = _load_label_volume(label_path)
                labels_local = np.asarray(labels_local)

                if labels_local.ndim != 3:
                    raise ValueError(f"Reference labels must be a 3D ZYX volume. Got shape={labels_local.shape}")

                if labels_local.shape != img_reference.shape:
                    raise ValueError(
                        "Reference labels and reference image must have the same shape. "
                        f"Image shape={img_reference.shape}; label shape={labels_local.shape}"
                    )

                img_seg_local = img_reference
                voxel_seg_local = voxel_reference
                segmentation_source = f"existing labels: {label_path}"

            else:
                chosen_method = seg_method_w.value
                settings = _build_segmentation_settings()

                if normalize_before_w.value:
                    if "ij_percentile_normalize" not in globals():
                        raise RuntimeError(
                            "normalize_before is enabled, but ij_percentile_normalize is not available."
                        )
                    print("\nPercentile-normalising reference volume before segmentation...")
                    img_seg_local = ij_percentile_normalize(img_reference)
                else:
                    img_seg_local = img_reference

                voxel_seg_local = voxel_reference

                print(f"\nRunning 2.5D segmentation on reference volume.")
                print(f"Method: {chosen_method}")
                print(f"Z-stitch IoU: {float(stitch_iou_w.value):.2f}")
                print(f"Settings: {settings}")

                labels_local = segment_nuclei_2p5d(
                    volume_zyx=img_seg_local,
                    method=chosen_method,
                    pixel_size_um_zyx=voxel_seg_local,
                    settings=settings,
                    min_iou=float(stitch_iou_w.value),
                    show_progress=True,
                )

                labels_local = np.asarray(labels_local)

                if labels_local.shape != img_seg_local.shape:
                    raise ValueError(
                        "Segmentation returned labels with unexpected shape. "
                        f"Image shape={img_seg_local.shape}; label shape={labels_local.shape}"
                    )

                segmentation_source = (
                    f"2.5D segmentation: method={chosen_method}, "
                    f"stitch_iou={float(stitch_iou_w.value):.2f}"
                )

                print("\nSaving reference label volume...")
                LABELS_FULL_PATH = SEG_DIR / "labels_full.tif"
                _save_tiff_zyx(
                    LABELS_FULL_PATH,
                    labels_local.astype(np.int32, copy=False),
                    voxel_size_um_zyx=voxel_seg_local,
                )

            if source_mode.value == "existing":
                # Still save a local copy of labels for reproducibility.
                LABELS_FULL_PATH = SEG_DIR / "labels_full.tif"
                print("\nSaving a local copy of the reference labels...")
                _save_tiff_zyx(
                    LABELS_FULL_PATH,
                    labels_local.astype(np.int32, copy=False),
                    voxel_size_um_zyx=voxel_seg_local,
                )

            print("\nExtracting 3D centroid and morphology features...")
            k_neighbors = int(k_neighbors_w.value)

            try:
                features_local = extract_nuclear_features_3d(
                    labels_local,
                    pixel_size_um=voxel_seg_local,
                    k_neighbors=k_neighbors,
                )
            except TypeError:
                # Some app versions do not expose k_neighbors as a keyword.
                features_local = extract_nuclear_features_3d(
                    labels_local,
                    pixel_size_um=voxel_seg_local,
                )

            features_local = pd.DataFrame(features_local)

            if len(features_local) == 0:
                raise RuntimeError("Feature extraction returned an empty table. Check segmentation quality.")

            FEATURES_FULL_PATH = SEG_DIR / "features_full.csv"
            features_local.to_csv(FEATURES_FULL_PATH, index=False)

            print(f"\nDetected 3D objects: {len(features_local)}")
            print(f"Saved labels:   {LABELS_FULL_PATH}")
            print(f"Saved features: {FEATURES_FULL_PATH}")

            # Apply globals for downstream 3D benchmark.
            img_full_seg = img_seg_local
            labels_full_seg = labels_local
            voxel_full_seg = voxel_seg_local
            df_full = features_local

            # Save/update metadata.
            metadata_path = result_dir / "inputs_metadata.json"
            metadata = {}

            if metadata_path.exists():
                try:
                    with open(metadata_path, "r") as f:
                        metadata = json.load(f)
                except Exception:
                    metadata = {}

            metadata.update({
                "full_image_path": globals().get("FULL_PATH_USED", globals().get("FULL_VOLUME_PATH", "")),
                "full_shape_zyx": list(img_reference.shape),
                "segmentation_shape_zyx": list(img_full_seg.shape),
                "voxel_size_full_um_zyx": list(voxel_reference),
                "voxel_size_full_seg_um_zyx": list(voxel_full_seg),
                "segmentation_source": segmentation_source,
                "labels_full_path": str(LABELS_FULL_PATH),
                "features_full_path": str(FEATURES_FULL_PATH),
                "n_reference_objects": int(len(df_full)),
                "k_neighbors_features": int(k_neighbors_w.value),
            })

            with open(metadata_path, "w") as f:
                json.dump(metadata, f, indent=2, default=str)

            print(f"Updated metadata: {metadata_path}")

        # Plot after processing succeeds.
        status_html.value = _status_ok_fn(
            [
                segmentation_source,
                f"Detected reference objects: {len(df_full)}",
                f"Saved labels to: {LABELS_FULL_PATH}",
                f"Saved features to: {FEATURES_FULL_PATH}",
                "Available globals: img_full_seg, labels_full_seg, voxel_full_seg, df_full",
            ],
            title="Reference segmentation complete",
        )

        with out_fig:
            _plot_reference_segmentation_overview(
                img_full_seg,
                labels_full_seg,
                df_full,
                title_prefix="Reference",
            )

        with out_tbl:
            display(widgets.HTML("<b>Reference feature table preview</b>"))
            display(df_full.head(20))

    except Exception as e:
        status_html.value = _status_err_fn([str(e)], title="Reference segmentation failed")

        with out_log:
            traceback.print_exc()


run_btn.on_click(on_run_reference_segmentation)

# ---------------------------------------------------------------------
# UI layout
# ---------------------------------------------------------------------

source_card = widgets.VBox([
    widgets.HTML("<b>Input source</b><br><span style='color:#64748B;'>Run 2.5D segmentation on the reference volume or load an existing 3D label mask.</span>"),
    source_mode,
    labels_path_w,
])

segmentation_card = widgets.VBox([
    widgets.HTML("<b>Segmentation settings</b>"),
    normalize_before_w,
    seg_method_w,
    inst_panel,
    thr_panel,
    stitch_iou_w,
])

feature_card = widgets.VBox([
    widgets.HTML("<b>Feature extraction</b>"),
    k_neighbors_w,
])

ui = widgets.VBox(
    [
        style_html,
        title_html,
        source_card,
        segmentation_card,
        feature_card,
        run_btn,
        widgets.Box([status_html], layout=widgets.Layout(width="100%", margin="10px 0")),
        out_log,
        out_fig,
        out_tbl,
    ],
    layout=widgets.Layout(width="95%"),
)

display(ui)

# 2. Run the 3D benchmark

In [ ]:
#@title Cell 2: Run the 3D benchmark

import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------
# 0. Check benchmark functions are available
# ---------------------------------------------------------

if "benchmark_patch_shapes_on_current_volume_multi_3d" not in globals():
    raise RuntimeError(
        "benchmark_patch_shapes_on_current_volume_multi_3d is not defined. "
        "Run Cell 1: 3D benchmark functions first."
    )

# ---------------------------------------------------------
# 1. Dynamic path setup
# ---------------------------------------------------------

if "RESULT_DIR" in globals() and RESULT_DIR:
    base_output_dir = Path(RESULT_DIR)
else:
    base_output_dir = Path("nucleisky3d_results")

base_output_dir.mkdir(parents=True, exist_ok=True)

output_dir_3d = base_output_dir / "benchmarking_3d"
output_dir_3d.mkdir(parents=True, exist_ok=True)

# Try to recover the original filename from metadata saved during Step 1.
base_name_3d = "current_volume"

try:
    meta_path = base_output_dir / "inputs_metadata.json"
    if meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)

        for key in ["full_image_path", "full_volume_path"]:
            if key in meta and meta[key]:
                base_name_3d = Path(meta[key]).stem
                break

except Exception:
    pass

# Fallback to globals if metadata did not exist.
if base_name_3d == "current_volume":
    if "FULL_PATH_USED" in globals() and FULL_PATH_USED:
        base_name_3d = Path(FULL_PATH_USED).stem
    elif "FULL_VOLUME_PATH" in globals() and FULL_VOLUME_PATH:
        base_name_3d = Path(FULL_VOLUME_PATH).stem

checkpoint_dir_3d = output_dir_3d / f"{base_name_3d}_checkpoints"

print(f"Saving 3D benchmark results to: {output_dir_3d}")
print(f"Checkpoint directory: {checkpoint_dir_3d}")
print(f"Base filename: {base_name_3d}")

# ---------------------------------------------------------
# 2. Prepare benchmark data
# ---------------------------------------------------------

# Prefer the segmentation-scale reference volume if Step 2 created it.
vol_for_bench_3d = globals().get("img_full_seg", globals().get("img_full", None))

# Prefer the segmentation-scale voxel size if Step 2 created it.
vox_for_bench_3d = globals().get("voxel_full_seg", globals().get("voxel_size_full_um", None))

if vol_for_bench_3d is None:
    raise RuntimeError(
        "Reference volume not found. Run Step 1 and Step 2 first. "
        "Expected img_full_seg or img_full."
    )

if "df_full" not in globals() or df_full is None:
    raise RuntimeError(
        "df_full not found. Run Step 2 to segment the reference volume and extract 3D features first."
    )

if vox_for_bench_3d is None:
    raise RuntimeError(
        "Voxel size not found. Run Step 1 and Step 2 first. "
        "Expected voxel_full_seg or voxel_size_full_um."
    )

vol_for_bench_3d = np.asarray(vol_for_bench_3d)

if vol_for_bench_3d.ndim != 3:
    raise ValueError(f"Expected a 3D ZYX reference volume. Got shape={vol_for_bench_3d.shape}")

vox_for_bench_3d = tuple(float(v) for v in vox_for_bench_3d)

if len(vox_for_bench_3d) != 3:
    raise ValueError("Voxel size must contain three values in Z,Y,X order.")

if any(v <= 0 for v in vox_for_bench_3d):
    raise ValueError("Voxel-size values must be positive.")

df_full_for_bench_3d = pd.DataFrame(df_full).copy()

required_any_centroids = (
    {"centroid_z_px", "centroid_y_px", "centroid_x_px"}.issubset(df_full_for_bench_3d.columns)
    or
    {"centroid_z_um", "centroid_y_um", "centroid_x_um"}.issubset(df_full_for_bench_3d.columns)
)

if not required_any_centroids:
    raise ValueError(
        "df_full must contain either pixel centroid columns "
        "(centroid_z_px, centroid_y_px, centroid_x_px) or micrometre centroid columns "
        "(centroid_z_um, centroid_y_um, centroid_x_um)."
    )

print("\nBenchmark input:")
print(f"  Volume shape ZYX: {tuple(vol_for_bench_3d.shape)}")
print(f"  Voxel size Z,Y,X: {vox_for_bench_3d} µm")
print(f"  Reference landmarks: {len(df_full_for_bench_3d)}")

# ---------------------------------------------------------
# 3. Benchmark settings
# ---------------------------------------------------------

# Patch shapes are Z,Y,X in pixels.
# Choose shapes according to your data anisotropy and expected query volume size.
PATCH_SHAPES_PX_ZYX_3D = [
    (32, 192, 192),
    (48, 256, 256),
    (64, 320, 320),
    (80, 384, 384),
    (96, 448, 448),
    (112, 512, 512),
    (128, 640, 640),
    (160, 768, 768),
    (192, 896, 896),
    (224, 1024, 1024),
]

PATCHES_PER_SHAPE_3D = 100
RANDOM_STATE_3D = 0
MIN_NUCLEI_PER_PATCH_3D = 3

MATCHERS_3D = (
    "pyramid",
    "hashing",
    "adaptive",
)

# Use defaults unless you want to customise.
matcher_config_3d = None

matcher_kwargs_map_3d = {
    "adaptive": {
        "matcher_order": ["pyramid", "hashing"],
        "stop_on_success": True,
        "max_total_time_s": None,
        "verbose": False,
    },

    # Optional examples:
    # "pyramid": {
    #     "_common": {
    #         "min_frac_inliers_success": 0.5,
    #     },
    #     "pyramid": {
    #         "n_iters": 50000,
    #     },
    # },
    # "hashing": {
    #     "hashing": {
    #         "max_candidates": 2000,
    #     },
    # },
}

RESTART_3D_BENCHMARK = False

print("\nBenchmark settings:")
print(f"  Patch shapes ZYX: {PATCH_SHAPES_PX_ZYX_3D}")
print(f"  Patches per shape: {PATCHES_PER_SHAPE_3D}")
print(f"  Matchers: {MATCHERS_3D}")
print(f"  Minimum landmarks per patch: {MIN_NUCLEI_PER_PATCH_3D}")
print(f"  Restart from scratch: {RESTART_3D_BENCHMARK}")

# ---------------------------------------------------------
# 4. Run benchmark
# ---------------------------------------------------------

all_results_3d, df_results_3d = benchmark_patch_shapes_on_current_volume_multi_3d(
    vol_full_zyx=vol_for_bench_3d,
    df_full=df_full_for_bench_3d,
    voxel_size_um_zyx=vox_for_bench_3d,

    patch_shapes_px_zyx=PATCH_SHAPES_PX_ZYX_3D,
    patches_per_shape=int(PATCHES_PER_SHAPE_3D),
    random_state=int(RANDOM_STATE_3D),

    matchers=MATCHERS_3D,

    matcher_config=matcher_config_3d,
    matcher_kwargs_map=matcher_kwargs_map_3d,

    save_path_dir=str(checkpoint_dir_3d),
    restart=bool(RESTART_3D_BENCHMARK),

    ij_percentile_normalize=ij_percentile_normalize,

    min_nuclei=int(MIN_NUCLEI_PER_PATCH_3D),
)

# ---------------------------------------------------------
# 5. Save final merged results
# ---------------------------------------------------------

csv_path_3d = output_dir_3d / f"{base_name_3d}_benchmark3d_results.csv"
df_results_3d.to_csv(csv_path_3d, index=False)

# Also save benchmark configuration for reproducibility.
config_path_3d = output_dir_3d / f"{base_name_3d}_benchmark3d_config.json"

config_3d = {
    "base_name": base_name_3d,
    "volume_shape_zyx": list(vol_for_bench_3d.shape),
    "voxel_size_um_zyx": list(vox_for_bench_3d),
    "n_reference_landmarks": int(len(df_full_for_bench_3d)),
    "patch_shapes_px_zyx": [list(x) for x in PATCH_SHAPES_PX_ZYX_3D],
    "patches_per_shape": int(PATCHES_PER_SHAPE_3D),
    "random_state": int(RANDOM_STATE_3D),
    "matchers": list(MATCHERS_3D),
    "min_nuclei_per_patch": int(MIN_NUCLEI_PER_PATCH_3D),
    "restart": bool(RESTART_3D_BENCHMARK),
    "checkpoint_dir": str(checkpoint_dir_3d),
    "results_csv": str(csv_path_3d),
}

with open(config_path_3d, "w") as f:
    json.dump(config_3d, f, indent=2)

print("\n3D benchmark complete.")
print(f"Saved final CSV: {csv_path_3d}")
print(f"Saved config:    {config_path_3d}")

print("\nResult preview:")
display(df_results_3d.head())

print("\nRows per matcher and patch shape:")
if len(df_results_3d) > 0:
    display(
        df_results_3d
        .groupby(["matcher", "patch_shape_label"], dropna=False)
        .size()
        .reset_index(name="n_rows")
        .sort_values(["patch_shape_label", "matcher"])
    )
else:
    print("No benchmark rows were generated.")

# 3. Plot results

In [ ]:
#@title Cell 3: Reload existing 3D benchmark results

import json
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------------------------------------
# 1. Infer likely default result path
# ---------------------------------------------------------

if "RESULT_DIR" in globals() and RESULT_DIR:
    base_output_dir = Path(RESULT_DIR)
else:
    base_output_dir = Path("nucleisky3d_results")

output_dir_3d = base_output_dir / "benchmarking_3d"

base_name_3d = globals().get("base_name_3d", "current_volume")

try:
    meta_path = base_output_dir / "inputs_metadata.json"
    if meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)

        for key in ["full_image_path", "full_volume_path"]:
            if key in meta and meta[key]:
                base_name_3d = Path(meta[key]).stem
                break

except Exception:
    pass

if base_name_3d == "current_volume":
    if "FULL_PATH_USED" in globals() and FULL_PATH_USED:
        base_name_3d = Path(FULL_PATH_USED).stem
    elif "FULL_VOLUME_PATH" in globals() and FULL_VOLUME_PATH:
        base_name_3d = Path(FULL_VOLUME_PATH).stem

default_csv_path_3d = output_dir_3d / f"{base_name_3d}_benchmark3d_results.csv"
default_config_path_3d = output_dir_3d / f"{base_name_3d}_benchmark3d_config.json"

# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def _load_3d_benchmark_csv(csv_path):
    """
    Load a 3D benchmark CSV and normalise key columns.
    """
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path)

    required = [
        "matcher",
        "patch_z_px",
        "patch_y_px",
        "patch_x_px",
        "patch_z0",
        "patch_y0",
        "patch_x0",
        "n_nuclei",
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            "This does not look like a 3D benchmark CSV. "
            f"Missing required columns: {missing}"
        )

    # Normalise matcher names.
    if "normalise_matcher_name_3d" in globals():
        df["matcher"] = df["matcher"].map(normalise_matcher_name_3d)
    elif "_normalise_matcher_name_3d" in globals():
        df["matcher"] = df["matcher"].map(_normalise_matcher_name_3d)
    else:
        df["matcher"] = df["matcher"].astype(str).str.strip().str.lower()

    # Ensure patch_shape_label exists.
    if "patch_shape_label" not in df.columns:
        df["patch_shape_label"] = (
            df["patch_z_px"].astype(int).astype(str)
            + "x"
            + df["patch_y_px"].astype(int).astype(str)
            + "x"
            + df["patch_x_px"].astype(int).astype(str)
        )

    # Convert commonly numeric columns.
    numeric_cols = [
        "patch_z_px",
        "patch_y_px",
        "patch_x_px",
        "patch_volume_px",
        "patch_xy_area_px",
        "patch_z0",
        "patch_y0",
        "patch_x0",
        "n_nuclei",
        "frac_inliers",
        "mean_error_um",
        "best_scale",
        "estimated_rotation_deg",
        "z_error_um",
        "y_error_um",
        "x_error_um",
        "trans_error_um",
        "z_error_px",
        "y_error_px",
        "x_error_px",
        "trans_error_px",
        "ssim_xy",
        "time_matcher_s",
        "time_total_s",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "success" in df.columns:
        df["success"] = (
            df["success"]
            .fillna(False)
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["true", "1", "yes", "y"])
        )

    return df


def _read_optional_config(config_path):
    config_path = Path(config_path)

    if not config_path.exists():
        return None

    try:
        with open(config_path, "r") as f:
            return json.load(f)
    except Exception:
        return None


def _summarise_loaded_3d_results(df):
    print(f"Loaded rows: {len(df)}")

    if len(df) == 0:
        return

    print("\nRows per matcher and patch shape:")
    display(
        df
        .groupby(["matcher", "patch_shape_label"], dropna=False)
        .size()
        .reset_index(name="n_rows")
        .sort_values(["patch_shape_label", "matcher"])
    )

    available_metrics = [
        c for c in [
            "success",
            "frac_inliers",
            "ssim_xy",
            "trans_error_px",
            "time_total_s",
        ]
        if c in df.columns
    ]

    if available_metrics:
        print("\nQuick numeric summary:")
        display(
            df[available_metrics]
            .describe(include="all")
        )


# ---------------------------------------------------------
# 3. UI
# ---------------------------------------------------------

already_loaded = "df_results_3d" in globals() and isinstance(globals().get("df_results_3d"), pd.DataFrame)

if already_loaded:
    print("df_results_3d is already present in the environment.")
    print(f"Rows: {len(df_results_3d)}")
    display(df_results_3d.head())

    reload_anyway = widgets.Checkbox(
        value=False,
        description="Reload from CSV anyway",
        indent=False,
    )
else:
    reload_anyway = widgets.Checkbox(
        value=True,
        description="Load from CSV",
        indent=False,
    )

csv_path_w = widgets.Text(
    value=str(default_csv_path_3d) if default_csv_path_3d.exists() else "",
    placeholder="e.g. /path/to/current_volume_benchmark3d_results.csv",
    description="CSV path:",
    layout=widgets.Layout(width="85%"),
)

config_path_w = widgets.Text(
    value=str(default_config_path_3d) if default_config_path_3d.exists() else "",
    placeholder="optional: /path/to/current_volume_benchmark3d_config.json",
    description="Config path:",
    layout=widgets.Layout(width="85%"),
)

load_btn = widgets.Button(
    description="Load 3D benchmark CSV",
    button_style="info",
    icon="upload",
)

out = widgets.Output()

for w in [csv_path_w, config_path_w]:
    try:
        w.style.description_width = "90px"
    except Exception:
        pass


def on_load_clicked(_):
    global df_results_3d, BENCHMARK3D_CSV_PATH, BENCHMARK3D_CONFIG
    global output_dir_3d, base_name_3d

    with out:
        clear_output()

        if not reload_anyway.value:
            print("Reload skipped.")
            return

        csv_path = csv_path_w.value.strip()

        if not csv_path:
            print("Please provide a CSV path.")
            return

        try:
            df_results_3d = _load_3d_benchmark_csv(csv_path)
            BENCHMARK3D_CSV_PATH = str(Path(csv_path))

            # Update output dir and base name from CSV path.
            csv_path_obj = Path(csv_path)
            output_dir_3d = csv_path_obj.parent
            base_name_3d = csv_path_obj.name.replace("_benchmark3d_results.csv", "")

            config_path = config_path_w.value.strip()
            if config_path:
                BENCHMARK3D_CONFIG = _read_optional_config(config_path)
            else:
                guessed_config = output_dir_3d / f"{base_name_3d}_benchmark3d_config.json"
                BENCHMARK3D_CONFIG = _read_optional_config(guessed_config)

            print(f"Successfully loaded df_results_3d from:\n  {BENCHMARK3D_CSV_PATH}")

            if BENCHMARK3D_CONFIG is not None:
                print("\nLoaded benchmark config:")
                display(BENCHMARK3D_CONFIG)
            else:
                print("\nNo benchmark config loaded.")

            print("\nPreview:")
            display(df_results_3d.head())

            _summarise_loaded_3d_results(df_results_3d)

        except Exception as e:
            print(f"Error loading 3D benchmark CSV: {e}")


load_btn.on_click(on_load_clicked)

display(
    widgets.VBox([
        reload_anyway,
        csv_path_w,
        config_path_w,
        load_btn,
        out,
    ])
)

# Auto-load if df_results_3d is absent and the inferred default path exists.
if (not already_loaded) and default_csv_path_3d.exists():
    with out:
        clear_output()
        try:
            df_results_3d = _load_3d_benchmark_csv(default_csv_path_3d)
            BENCHMARK3D_CSV_PATH = str(default_csv_path_3d)
            BENCHMARK3D_CONFIG = _read_optional_config(default_config_path_3d)

            print(f"Auto-loaded df_results_3d from:\n  {BENCHMARK3D_CSV_PATH}")
            display(df_results_3d.head())
            _summarise_loaded_3d_results(df_results_3d)

        except Exception as e:
            print(f"Auto-load failed: {e}")

In [ ]:
#@title Cell 4: Plot 3D benchmark results

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import curve_fit
from pathlib import Path
from IPython.display import display, Markdown

# ---------------------------------------------------------
# 0. Plotting configuration
# ---------------------------------------------------------

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"

# Output directory
if "output_dir_3d" in globals():
    plot_output_dir_3d = Path(output_dir_3d) / "plots"
elif "RESULT_DIR" in globals() and RESULT_DIR:
    plot_output_dir_3d = Path(RESULT_DIR) / "benchmarking_3d" / "plots"
else:
    plot_output_dir_3d = Path("nucleisky3d_benchmark_plots")

plot_output_dir_3d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Success criteria for 3D benchmark
# ---------------------------------------------------------

# For 3D, translation error is essential because SSIM is computed only on XY MIPs.
TARGET_PROB_3D = 0.90

FRAC_INLIERS_THRESH_3D = 0.50
ERROR_PX_THRESH_3D = 20.0

# SSIM on XY MIPs can be useful, but should not be the only success criterion.
# Set USE_SSIM_XY_FOR_SUCCESS = False if you want a geometry-only success definition.
USE_SSIM_XY_FOR_SUCCESS = True
SSIM_XY_THRESH_3D = 0.80

# ---------------------------------------------------------
# 2. Colours and matcher names
# ---------------------------------------------------------

PREFERRED_MATCHER_ORDER_3D = [
    "pyramid",
    "hashing",
    "adaptive",
]

FIXED_MATCHER_COLORS_3D = {
    "pyramid": "#CC79A7",   # purple
    "hashing": "#009E73",   # green
    "adaptive": "#0072B2",  # blue
}


def normalise_matcher_name_3d_plot(matcher):
    matcher = str(matcher).strip().lower()

    aliases = {
        "hash": "hashing",
        "hashing3d": "hashing",
        "geometric_hashing": "hashing",
        "geometric hashing": "hashing",
        "pyramid3d": "pyramid",
        "tetrahedron": "pyramid",
        "tetrahedra": "pyramid",
        "adaptive3d": "adaptive",
    }

    return aliases.get(matcher, matcher)


def get_matcher_order_3d(df):
    present = sorted(df["matcher"].dropna().map(normalise_matcher_name_3d_plot).unique())
    ordered = [m for m in PREFERRED_MATCHER_ORDER_3D if m in present]
    ordered += [m for m in present if m not in ordered]
    return ordered


def make_matcher_color_map_3d(matchers):
    color_map = {}

    for matcher in matchers:
        if matcher in FIXED_MATCHER_COLORS_3D:
            color_map[matcher] = FIXED_MATCHER_COLORS_3D[matcher]

    unknown = [m for m in matchers if m not in color_map]
    if unknown:
        fallback = plt.get_cmap("tab10")
        for i, matcher in enumerate(unknown):
            color_map[matcher] = fallback(i % 10)

    return color_map


# ---------------------------------------------------------
# 3. Data preparation
# ---------------------------------------------------------

def sanitize_and_score_3d(df_results_3d):
    """
    Prepare the 3D benchmark dataframe and define is_success.

    Success definition:
      - matcher-reported success is True
      - inlier fraction >= FRAC_INLIERS_THRESH_3D
      - translation error < ERROR_PX_THRESH_3D
      - optionally, XY-MIP SSIM >= SSIM_XY_THRESH_3D

    Important:
    ssim_xy is an XY-MIP diagnostic, not full volumetric SSIM.
    """
    df = pd.DataFrame(df_results_3d).copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x) for x in t if str(x) != ""]).strip()
            for t in df.columns
        ]

    df.columns = [str(c).strip() for c in df.columns]

    required = [
        "matcher",
        "patch_z_px",
        "patch_y_px",
        "patch_x_px",
        "n_nuclei",
        "success",
        "frac_inliers",
        "trans_error_px",
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"df_results_3d is missing required columns: {missing}")

    df["matcher"] = df["matcher"].map(normalise_matcher_name_3d_plot)

    if "patch_shape_label" not in df.columns:
        df["patch_shape_label"] = (
            df["patch_z_px"].astype(int).astype(str)
            + "x"
            + df["patch_y_px"].astype(int).astype(str)
            + "x"
            + df["patch_x_px"].astype(int).astype(str)
        )

    numeric_cols = [
        "patch_z_px",
        "patch_y_px",
        "patch_x_px",
        "patch_volume_px",
        "patch_xy_area_px",
        "patch_z0",
        "patch_y0",
        "patch_x0",
        "n_nuclei",
        "frac_inliers",
        "mean_error_um",
        "best_scale",
        "estimated_rotation_deg",
        "z_error_um",
        "y_error_um",
        "x_error_um",
        "trans_error_um",
        "z_error_px",
        "y_error_px",
        "x_error_px",
        "trans_error_px",
        "ssim_xy",
        "time_matcher_s",
        "time_total_s",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Robust bool parsing.
    df["success"] = (
        df["success"]
        .fillna(False)
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )

    df["n_nuclei"] = pd.to_numeric(df["n_nuclei"], errors="coerce").fillna(0).astype(int)

    is_success = (
        (df["success"] == True) &
        (df["frac_inliers"] >= FRAC_INLIERS_THRESH_3D) &
        (df["trans_error_px"] < ERROR_PX_THRESH_3D)
    )

    if USE_SSIM_XY_FOR_SUCCESS:
        if "ssim_xy" not in df.columns:
            raise ValueError("USE_SSIM_XY_FOR_SUCCESS=True, but df_results_3d has no ssim_xy column.")
        is_success = is_success & (df["ssim_xy"] >= SSIM_XY_THRESH_3D)

    df["is_success"] = is_success.fillna(False)

    # Absolute errors are often easier to plot.
    for axis in ["z", "y", "x"]:
        col = f"{axis}_error_px"
        if col in df.columns:
            df[f"abs_{axis}_error_px"] = np.abs(pd.to_numeric(df[col], errors="coerce"))

        col_um = f"{axis}_error_um"
        if col_um in df.columns:
            df[f"abs_{axis}_error_um"] = np.abs(pd.to_numeric(df[col_um], errors="coerce"))

    return df


def wilson_interval(k, n, z=1.96):
    """
    95% Wilson confidence interval for a binomial proportion.
    """
    if n <= 0:
        return np.nan, np.nan

    p_hat = k / n
    denom = 1 + z**2 / n
    centre = (p_hat + z**2 / (2 * n)) / denom
    half_width = z * np.sqrt(
        p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)
    ) / denom

    return max(0.0, centre - half_width), min(1.0, centre + half_width)


# ---------------------------------------------------------
# 4. Modelling and binning
# ---------------------------------------------------------

def logistic_log_n_3d(log_n, k, log_n0):
    return 1.0 / (1.0 + np.exp(-k * (log_n - log_n0)))


def nuclei_for_target_from_logistic_3d(k, log_n0, target=0.90):
    if k <= 0 or target <= 0 or target >= 1:
        return np.nan

    log_n_target = log_n0 - np.log((1.0 / target) - 1.0) / k
    return 10 ** log_n_target


def make_log_nuclei_bins_3d(n_values):
    """
    Bins for 3D query landmark counts.
    """
    n_values = np.asarray(n_values, dtype=float)
    n_values = n_values[np.isfinite(n_values) & (n_values > 0)]

    if len(n_values) == 0:
        raise ValueError("No positive landmark counts available.")

    max_n = int(np.nanmax(n_values))

    low_edges = np.array([1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100])

    if max_n <= 100:
        edges = np.arange(1, max_n + 2, 1)
    else:
        high_edges = np.unique(
            np.round(
                np.logspace(np.log10(100), np.log10(max_n + 1), 12)
            ).astype(int)
        )

        edges = np.unique(np.concatenate([low_edges, high_edges]))
        edges = edges[(edges >= 1) & (edges <= max_n + 1)]

    if len(edges) < 3:
        edges = np.array([1, max_n + 1])

    return edges


def empirical_threshold_from_bins_3d(binned, target=0.90, require_sustained=True):
    b = binned.sort_values("median_nuclei").copy()

    if len(b) == 0:
        return np.nan

    y = b["success_probability"].to_numpy(dtype=float)
    x = b["median_nuclei"].to_numpy(dtype=float)

    if require_sustained:
        for i in range(len(b)):
            if np.all(y[i:] >= target):
                return float(x[i])
        return np.nan

    above = b[y >= target]

    if len(above) == 0:
        return np.nan

    return float(above["median_nuclei"].iloc[0])


# ---------------------------------------------------------
# 5. Plot functions
# ---------------------------------------------------------

def plot_3d_success_vs_nuclei(
    df,
    matcher_order,
    color_map,
    target=0.90,
    save_path=None,
    save_table_path=None,
):
    """
    Plot probability of correct 3D subvolume localisation versus number of landmarks.
    """
    d = df.copy()
    d = d[np.isfinite(d["n_nuclei"]) & (d["n_nuclei"] > 0)].copy()

    if d.empty:
        raise ValueError("No rows with n_nuclei > 0 available.")

    fig, ax = plt.subplots(figsize=(7.4, 4.9), dpi=150)

    summary_rows = []

    for matcher in matcher_order:
        if matcher not in set(d["matcher"].dropna().unique()):
            continue

        g = d[d["matcher"] == matcher].copy()
        color = color_map[matcher]

        bin_edges = make_log_nuclei_bins_3d(g["n_nuclei"].values)

        g["nuclei_bin"] = pd.cut(
            g["n_nuclei"],
            bins=bin_edges,
            include_lowest=True,
            duplicates="drop",
        )

        binned = (
            g
            .groupby("nuclei_bin", observed=True)
            .agg(
                n_trials=("is_success", "size"),
                n_success=("is_success", "sum"),
                median_nuclei=("n_nuclei", "median"),
                mean_nuclei=("n_nuclei", "mean"),
                success_probability=("is_success", "mean"),
            )
            .reset_index()
            .dropna(subset=["median_nuclei", "success_probability"])
        )

        if len(binned) == 0:
            summary_rows.append({
                "matcher": matcher,
                "n_trials": len(g),
                f"empirical_min_landmarks_{int(target * 100)}": np.nan,
                f"fit_min_landmarks_{int(target * 100)}": np.nan,
                "fit_status": "no binned data",
            })
            continue

        ci = binned.apply(
            lambda r: wilson_interval(r["n_success"], r["n_trials"]),
            axis=1,
            result_type="expand",
        )

        binned["ci_low"] = ci[0]
        binned["ci_high"] = ci[1]

        ax.plot(
            binned["median_nuclei"],
            binned["success_probability"],
            "o",
            color=color,
            alpha=0.9,
            markersize=5,
            label=f"{matcher} observed",
        )

        ax.fill_between(
            binned["median_nuclei"].to_numpy(dtype=float),
            binned["ci_low"].to_numpy(dtype=float),
            binned["ci_high"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=0,
        )

        empirical_n = empirical_threshold_from_bins_3d(
            binned,
            target=target,
            require_sustained=True,
        )

        fit_n = np.nan
        fit_status = "fit failed"

        try:
            x_raw = g["n_nuclei"].to_numpy(dtype=float)
            y_raw = g["is_success"].to_numpy(dtype=float)

            good = np.isfinite(x_raw) & (x_raw > 0) & np.isfinite(y_raw)
            x_raw = x_raw[good]
            y_raw = y_raw[good]

            if len(np.unique(y_raw)) < 2:
                fit_status = "all successes or all failures"
            else:
                x_log = np.log10(x_raw)

                p0 = [4.0, np.nanmedian(x_log)]
                bounds = ([0.01, np.nanmin(x_log)], [50.0, np.nanmax(x_log)])

                popt, _ = curve_fit(
                    logistic_log_n_3d,
                    x_log,
                    y_raw,
                    p0=p0,
                    bounds=bounds,
                    maxfev=20000,
                )

                k, log_n0 = popt
                fit_n = nuclei_for_target_from_logistic_3d(k, log_n0, target=target)

                x_fit = np.logspace(
                    np.log10(np.nanmin(x_raw)),
                    np.log10(np.nanmax(x_raw)),
                    300,
                )
                y_fit = logistic_log_n_3d(np.log10(x_fit), k, log_n0)

                ax.plot(
                    x_fit,
                    y_fit,
                    color=color,
                    linewidth=2.0,
                    alpha=0.85,
                    label=f"{matcher} fit",
                )

                fit_status = "ok"

        except Exception as e:
            fit_status = f"fit failed: {type(e).__name__}"

        summary_rows.append({
            "matcher": matcher,
            "n_trials": len(g),
            f"empirical_min_landmarks_{int(target * 100)}": empirical_n,
            f"fit_min_landmarks_{int(target * 100)}": fit_n,
            "fit_status": fit_status,
        })

    ax.axhline(
        target,
        color="0.35",
        linestyle="--",
        linewidth=1.0,
        alpha=0.9,
    )

    ax.text(
        0.02,
        target + 0.035,
        f"{int(target * 100)}% success",
        transform=ax.get_yaxis_transform(),
        fontsize=9,
        color="0.3",
    )

    ax.set_xscale("log")

    max_n = int(np.nanmax(d["n_nuclei"]))
    min_n = max(1, int(np.nanmin(d["n_nuclei"])))

    ax.set_xlim(max(1, min_n * 0.85), max_n * 1.15)
    ax.set_ylim(-0.04, 1.04)

    candidate_ticks = np.array([
        1, 2, 3, 5, 10, 20, 50,
        100, 200, 500, 1000, 2000, 5000,
        10000, 20000,
    ])

    ticks = candidate_ticks[
        (candidate_ticks >= max(1, min_n)) &
        (candidate_ticks <= max_n * 1.15)
    ]

    ax.set_xticks(ticks)
    ax.set_xticklabels([str(int(t)) for t in ticks])

    ax.set_xlabel("Number of landmarks in query subvolume")
    ax.set_ylabel("Probability of correct 3D localisation")
    ax.set_title("3D localisation success versus landmark count")

    ax.grid(True, which="both", alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches="tight", transparent=True)
        print(f"Saved: {save_path}")

    summary = pd.DataFrame(summary_rows)

    sort_col = f"empirical_min_landmarks_{int(target * 100)}"
    if sort_col in summary.columns:
        summary = summary.sort_values(sort_col, na_position="last")

    if save_table_path:
        summary.to_csv(save_table_path, index=False)
        print(f"Saved table: {save_table_path}")

    plt.show()

    return summary


def plot_3d_success_vs_patch_shape(
    df,
    matcher_order,
    color_map,
    save_path=None,
):
    """
    Plot success probability by 3D patch shape.
    Categorical spacing is clearer than treating ZYX shape as a scalar.
    """
    d = df.copy()

    # Preserve shape order by patch volume and then Z/Y/X.
    shape_table = (
        d[["patch_shape_label", "patch_z_px", "patch_y_px", "patch_x_px"]]
        .drop_duplicates()
        .copy()
    )

    shape_table["patch_volume_px"] = (
        shape_table["patch_z_px"] *
        shape_table["patch_y_px"] *
        shape_table["patch_x_px"]
    )

    shape_table = shape_table.sort_values(
        ["patch_volume_px", "patch_z_px", "patch_y_px", "patch_x_px"]
    )

    shape_order = shape_table["patch_shape_label"].tolist()
    shape_to_x = {s: i for i, s in enumerate(shape_order)}

    fig, ax = plt.subplots(figsize=(7.8, 4.8), dpi=150)

    grouped = (
        d
        .groupby(["matcher", "patch_shape_label"], observed=True)
        .agg(
            n_trials=("is_success", "size"),
            n_success=("is_success", "sum"),
            success_probability=("is_success", "mean"),
        )
        .reset_index()
    )

    for matcher in matcher_order:
        if matcher not in set(grouped["matcher"].dropna().unique()):
            continue

        g = grouped[grouped["matcher"] == matcher].copy()
        g["x"] = g["patch_shape_label"].map(shape_to_x)
        g = g.sort_values("x")

        ci = g.apply(
            lambda r: wilson_interval(r["n_success"], r["n_trials"]),
            axis=1,
            result_type="expand",
        )

        g["ci_low"] = ci[0]
        g["ci_high"] = ci[1]

        color = color_map[matcher]

        ax.plot(
            g["x"],
            g["success_probability"],
            "o-",
            color=color,
            linewidth=2.0,
            markersize=5,
            label=matcher,
        )

        ax.fill_between(
            g["x"].to_numpy(dtype=float),
            g["ci_low"].to_numpy(dtype=float),
            g["ci_high"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=0,
        )

    ax.set_xticks(range(len(shape_order)))
    ax.set_xticklabels(shape_order, rotation=35, ha="right")

    ax.set_ylim(-0.04, 1.04)
    ax.set_xlabel("Query subvolume size, Z×Y×X pixels")
    ax.set_ylabel("Probability of correct 3D localisation")
    ax.set_title("3D localisation success versus query subvolume size")

    ax.grid(True, alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches="tight", transparent=True)
        print(f"Saved: {save_path}")

    plt.show()


def plot_3d_runtime_vs_nuclei(
    df,
    matcher_order,
    color_map,
    save_path=None,
):
    """
    Plot mean runtime versus landmark count.
    """
    if "time_total_s" not in df.columns:
        print("time_total_s not found; skipping runtime plot.")
        return

    d = df.copy()
    d = d[np.isfinite(d["n_nuclei"]) & (d["n_nuclei"] > 0)].copy()

    if d.empty:
        print("No positive n_nuclei rows available for runtime plot.")
        return

    fig, ax = plt.subplots(figsize=(7.4, 4.9), dpi=150)

    for matcher in matcher_order:
        if matcher not in set(d["matcher"].dropna().unique()):
            continue

        g = d[d["matcher"] == matcher].copy()
        color = color_map[matcher]

        bin_edges = make_log_nuclei_bins_3d(g["n_nuclei"].values)

        g["nuclei_bin"] = pd.cut(
            g["n_nuclei"],
            bins=bin_edges,
            include_lowest=True,
            duplicates="drop",
        )

        binned = (
            g
            .groupby("nuclei_bin", observed=True)
            .agg(
                median_nuclei=("n_nuclei", "median"),
                mean_total_time_s=("time_total_s", "mean"),
                median_total_time_s=("time_total_s", "median"),
            )
            .reset_index()
            .dropna(subset=["median_nuclei", "mean_total_time_s"])
            .sort_values("median_nuclei")
        )

        ax.plot(
            binned["median_nuclei"],
            binned["mean_total_time_s"],
            "o-",
            color=color,
            linewidth=2.0,
            markersize=5,
            label=matcher,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel("Number of landmarks in query subvolume")
    ax.set_ylabel("Mean total runtime per query (s)")
    ax.set_title("3D benchmark runtime versus landmark count")

    ax.grid(True, which="both", alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches="tight", transparent=True)
        print(f"Saved: {save_path}")

    plt.show()


def plot_3d_axis_errors(
    df,
    matcher_order,
    color_map,
    save_path=None,
):
    """
    Plot median absolute translation error along Z, Y and X.

    This is useful because 3D microscopy data are often anisotropic and Z errors
    may behave differently from XY errors.
    """
    required = ["abs_z_error_px", "abs_y_error_px", "abs_x_error_px"]

    if not all(c in df.columns for c in required):
        print("Per-axis error columns not found; skipping axis-error plot.")
        return

    d = df.copy()

    # Only rows where a transform was produced.
    d = d[np.isfinite(d["trans_error_px"])].copy()

    if d.empty:
        print("No rows with finite translation error; skipping axis-error plot.")
        return

    shape_table = (
        d[["patch_shape_label", "patch_z_px", "patch_y_px", "patch_x_px"]]
        .drop_duplicates()
        .copy()
    )

    shape_table["patch_volume_px"] = (
        shape_table["patch_z_px"] *
        shape_table["patch_y_px"] *
        shape_table["patch_x_px"]
    )

    shape_table = shape_table.sort_values(
        ["patch_volume_px", "patch_z_px", "patch_y_px", "patch_x_px"]
    )

    shape_order = shape_table["patch_shape_label"].tolist()
    shape_to_x = {s: i for i, s in enumerate(shape_order)}

    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.2), dpi=150, sharey=True)

    axis_info = [
        ("abs_z_error_px", "Z error"),
        ("abs_y_error_px", "Y error"),
        ("abs_x_error_px", "X error"),
    ]

    for ax, (col, title) in zip(axes, axis_info):
        grouped = (
            d
            .groupby(["matcher", "patch_shape_label"], observed=True)
            .agg(
                median_error=(col, "median"),
                q25=(col, lambda x: np.nanpercentile(x, 25)),
                q75=(col, lambda x: np.nanpercentile(x, 75)),
            )
            .reset_index()
        )

        for matcher in matcher_order:
            if matcher not in set(grouped["matcher"].dropna().unique()):
                continue

            g = grouped[grouped["matcher"] == matcher].copy()
            g["x"] = g["patch_shape_label"].map(shape_to_x)
            g = g.sort_values("x")

            color = color_map[matcher]

            ax.plot(
                g["x"],
                g["median_error"],
                "o-",
                color=color,
                linewidth=2.0,
                markersize=5,
                label=matcher,
            )

            ax.fill_between(
                g["x"].to_numpy(dtype=float),
                g["q25"].to_numpy(dtype=float),
                g["q75"].to_numpy(dtype=float),
                color=color,
                alpha=0.15,
                linewidth=0,
            )

        ax.set_xticks(range(len(shape_order)))
        ax.set_xticklabels(shape_order, rotation=35, ha="right")
        ax.set_title(title)
        ax.set_xlabel("Query subvolume size, Z×Y×X pixels")
        ax.grid(True, alpha=0.25)

    axes[0].set_ylabel("Median absolute translation error (pixels)")
    axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    fig.suptitle("Per-axis translation error in 3D benchmark", y=1.04, fontsize=13)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, bbox_inches="tight", transparent=True)
        print(f"Saved: {save_path}")

    plt.show()


# ---------------------------------------------------------
# 6. Execution
# ---------------------------------------------------------

if "df_results_3d" not in globals():
    raise RuntimeError(
        "df_results_3d is not defined. Run the 3D benchmark or the reload cell first."
    )

df_clean_3d = sanitize_and_score_3d(df_results_3d)

MATCHER_ORDER_3D = get_matcher_order_3d(df_clean_3d)
MATCHER_COLOR_MAP_3D = make_matcher_color_map_3d(MATCHER_ORDER_3D)

print(f"Total evaluated 3D patches: {len(df_clean_3d)}")
print("\n3D success criteria:")
print(f"  matcher-reported success = True")
print(f"  inlier fraction >= {FRAC_INLIERS_THRESH_3D}")
print(f"  translation error < {ERROR_PX_THRESH_3D} px")
if USE_SSIM_XY_FOR_SUCCESS:
    print(f"  XY-MIP SSIM >= {SSIM_XY_THRESH_3D}")
else:
    print("  XY-MIP SSIM used as diagnostic only")

print("\nConsistent 3D matcher colours:")
for matcher in MATCHER_ORDER_3D:
    print(f"  {matcher}: {MATCHER_COLOR_MAP_3D[matcher]}")

display(Markdown(
    "### 3D benchmark summary\n"
    "Success is defined using matcher success, inlier fraction and translation error. "
    "When enabled, XY-MIP SSIM is also required, but it should be interpreted as an image-level diagnostic rather than full volumetric SSIM. "
    "Per-axis error plots are included because Z and XY localisation errors can behave differently in anisotropic microscopy volumes."
))

# Summary table
summary_3d = (
    df_clean_3d
    .groupby(["matcher", "patch_shape_label"], observed=True)
    .agg(
        n_trials=("is_success", "size"),
        n_success=("is_success", "sum"),
        success_probability=("is_success", "mean"),
        median_nuclei=("n_nuclei", "median"),
        median_trans_error_px=("trans_error_px", "median"),
        median_time_total_s=("time_total_s", "median"),
    )
    .reset_index()
    .sort_values(["patch_shape_label", "matcher"])
)

summary_path_3d = plot_output_dir_3d / "benchmark3d_summary_by_patch_shape.csv"
summary_3d.to_csv(summary_path_3d, index=False)

print(f"\nSaved summary table: {summary_path_3d}")

display(
    summary_3d
    .style
    .format({
        "success_probability": "{:.2f}",
        "median_nuclei": "{:.1f}",
        "median_trans_error_px": "{:.2f}",
        "median_time_total_s": "{:.2f}",
    })
    .background_gradient(
        subset=["success_probability"],
        cmap="Greens",
        vmin=0,
        vmax=1,
    )
)

# Plot 1: success vs landmark count
threshold_table_3d = plot_3d_success_vs_nuclei(
    df_clean_3d,
    matcher_order=MATCHER_ORDER_3D,
    color_map=MATCHER_COLOR_MAP_3D,
    target=TARGET_PROB_3D,
    save_path=plot_output_dir_3d / "benchmark3d_success_vs_landmark_count.pdf",
    save_table_path=plot_output_dir_3d / "benchmark3d_landmark_thresholds.csv",
)

print(f"\nEstimated landmark count required for {int(TARGET_PROB_3D * 100)}% localisation probability:")
display(
    threshold_table_3d
    .style
    .format({
        f"empirical_min_landmarks_{int(TARGET_PROB_3D * 100)}": "{:.1f}",
        f"fit_min_landmarks_{int(TARGET_PROB_3D * 100)}": "{:.1f}",
    })
)

# Plot 2: success vs patch shape
plot_3d_success_vs_patch_shape(
    df_clean_3d,
    matcher_order=MATCHER_ORDER_3D,
    color_map=MATCHER_COLOR_MAP_3D,
    save_path=plot_output_dir_3d / "benchmark3d_success_vs_patch_shape.pdf",
)

# Plot 3: runtime
plot_3d_runtime_vs_nuclei(
    df_clean_3d,
    matcher_order=MATCHER_ORDER_3D,
    color_map=MATCHER_COLOR_MAP_3D,
    save_path=plot_output_dir_3d / "benchmark3d_runtime_vs_landmark_count.pdf",
)

# Plot 4: axis errors
plot_3d_axis_errors(
    df_clean_3d,
    matcher_order=MATCHER_ORDER_3D,
    color_map=MATCHER_COLOR_MAP_3D,
    save_path=plot_output_dir_3d / "benchmark3d_axis_translation_errors.pdf",
)